# 1\.2\.4 Create Harmonized Mobility Tables

At this point in the pipeline, all mobility datasets have undergone source staging, temporal standardization, and spatial standardization where applicable\. This notebook serves as the final harmonization step before multimodal integration\. The primary objectives are to resolve the remaining issues deferred from earlier processing stages, align all datasets to the common study period, validate known spatial exceptions, and write finalized harmonized outputs for downstream Taxi Zone × Date × Temporal Bucket aggregation\. By addressing these final cleanup tasks here, future notebooks can consume a stable, integration\-ready layer rather than repeatedly applying study\-period filters, spatial fixes, or source\-specific exclusions\.

In [1]:
!pip install altair==6.1.0


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [2]:
from pathlib import Path

import re
import gc
import time
import shutil

import duckdb
import pandas as pd
import json

import altair as alt

from project_branding import BRAND_COLORS

We have to do several expensive harmonization and aggregation steps\. Use the toggles below to control which mobility tables are rebuilt during execution\. In most cases, previously generated outputs can be reused unless a source dataset or transformation logic has changed\.

In [3]:
# ---------------------------------------------------------------------
# 1.2.4 Control Panel and Paths
# ---------------------------------------------------------------------

PIPELINE_DATA_DIR = Path("pipeline_data")

# Rebuild expensive QA manifests
REBUILD_HARMONIZATION_INVENTORY = False
REBUILD_TLC_SOURCE_QA = False
REBUILD_TLC_SIZE_REDUCTION_QA = False

# Rebuild harmonized outputs
REBUILD_HARMONIZED_BUS = False
REBUILD_HARMONIZED_TRAFFIC = False
REBUILD_HARMONIZED_SUBWAY = False
REBUILD_HARMONIZED_TLC = False
REBUILD_HARMONIZED_BRIDGE_TUNNEL = False
REBUILD_HARMONIZED_TAXI_ZONES = False

# Rebuild TLC chunked intermediate outputs
REBUILD_TLC_CHUNK_PLAN = False
REBUILD_TLC_CHUNKS = False
REBUILD_TLC_FINAL_DROPOFF_CHUNKS = False
REBUILD_TLC_QUARTER_FINAL_CHUNKS = False

# Rebuild final validation summaries
REBUILD_FINAL_HARMONIZATION_QA = False

# # For rebuilds
# REBUILD_HARMONIZATION_INVENTORY = True
# REBUILD_TLC_SOURCE_QA = True
# REBUILD_TLC_SIZE_REDUCTION_QA = True

# REBUILD_HARMONIZED_TLC = True

# REBUILD_TLC_CHUNK_PLAN = True
# REBUILD_TLC_CHUNKS = True
# REBUILD_TLC_FINAL_DROPOFF_CHUNKS = True
# REBUILD_TLC_QUARTER_FINAL_CHUNKS = True

# REBUILD_FINAL_HARMONIZATION_QA = True
# ## End of rebuild toggles

MIN_VALID_TLC_TRIP_DURATION_SECONDS = 60
MAX_VALID_TLC_TRIP_DURATION_SECONDS = 86_400
MIN_VALID_TLC_TRIP_DISTANCE = 0
MAX_VALID_TLC_TRIP_DISTANCE = 100
MAX_VALID_TLC_TRIP_SPEED_MPH = 100

# Source paths
traffic_spatiotemporal_path = (
    PIPELINE_DATA_DIR / "1.2.2.traffic_with_zones.parquet"
)

bus_spatiotemporal_glob = (
    PIPELINE_DATA_DIR / "1.2.2.bus_speeds_spatiotemporal_parquet" / "*.parquet"
)

subway_spatiotemporal_glob = (
    PIPELINE_DATA_DIR / "1.2.2.subway_with_zones_parquet" / "*.parquet"
)

tlc_mobility_glob = (
    PIPELINE_DATA_DIR / "1.2.3.final_tables" / "*.parquet"
)

bridge_tunnel_temporal_path = (
    PIPELINE_DATA_DIR / "1.2.1.bridge_tunnel_temporal.parquet"
)

taxi_zones_path = (
    PIPELINE_DATA_DIR / "1.1.6.nyc_taxi_zones.parquet"
)

# Output paths
HARMONIZED_OUTPUT_DIR = (
    PIPELINE_DATA_DIR / "1.2.4.harmonized_mobility_datasets"
)

MANIFEST_DIR = (
    PIPELINE_DATA_DIR / "1.2.4.manifests"
)

HARMONIZED_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

harmonized_bus_path = HARMONIZED_OUTPUT_DIR / "bus_mobility_table.parquet"
harmonized_traffic_path = HARMONIZED_OUTPUT_DIR / "traffic_mobility_table.parquet"
harmonized_subway_path = HARMONIZED_OUTPUT_DIR / "subway_mobility_table.parquet"
harmonized_tlc_path = HARMONIZED_OUTPUT_DIR / "tlc_mobility_table.parquet"
harmonized_bridge_tunnel_path = HARMONIZED_OUTPUT_DIR / "bridge_tunnel_mobility_table.parquet"
harmonized_taxi_zones_path = HARMONIZED_OUTPUT_DIR / "nyc_taxi_zones_harmonized.parquet"

# TLC intermediate paths
tlc_chunk_dir = HARMONIZED_OUTPUT_DIR / "tlc_chunked_intermediate"
tlc_final_dropoff_chunk_dir = HARMONIZED_OUTPUT_DIR / "tlc_final_dropoff_chunks"
tlc_quarter_final_dir = HARMONIZED_OUTPUT_DIR / "tlc_quarter_final_chunks"

tlc_chunk_dir.mkdir(parents=True, exist_ok=True)
tlc_final_dropoff_chunk_dir.mkdir(parents=True, exist_ok=True)
tlc_quarter_final_dir.mkdir(parents=True, exist_ok=True)

The project requires a common temporal window across all mobility datasets\. The parameters below define the study period that will be used when constructing the final harmonized mobility tables\.

In [4]:
# ---------------------------------------------------------------------
# 1.2.4 Study Period Controls
# ---------------------------------------------------------------------

STUDY_START_DATE = "2023-01-01"
STUDY_END_DATE = "2026-03-31"

## 1\.2\.4\.1 Inspect Harmonized Mobility Datasets

This section inspects the standardized mobility datasets that will feed the final harmonization process\. Because several of these datasets are too large to load into memory at the same time, we use DuckDB to query parquet metadata and summary statistics directly from disk\. The goal is to establish each dataset’s current row count, date coverage, schema, spatial identifier, and primary mobility measure before applying study\-period filters or remaining harmonization fixes\.

First, confirm that each expected standardized input exists\. For large partitioned datasets, we check the parent directory and count parquet files rather than loading the full dataset\.

In [5]:
# ---------------------------------------------------------------------
# 1.2.4.1 Inspect Harmonized Mobility Datasets
# ---------------------------------------------------------------------

single_file_inputs = {
    "traffic": traffic_spatiotemporal_path,
    "bridge_tunnel": bridge_tunnel_temporal_path,
    "taxi_zones": taxi_zones_path,
}

partitioned_inputs = {
    "bus": bus_spatiotemporal_glob,
    "subway": subway_spatiotemporal_glob,
    "tlc": tlc_mobility_glob,
}

input_path_checks = []

for dataset_name, path in single_file_inputs.items():
    input_path_checks.append({
        "dataset": dataset_name,
        "path": str(path),
        "path_type": "single_file",
        "exists": path.exists(),
        "file_count": 1 if path.exists() else 0
    })

for dataset_name, glob_path in partitioned_inputs.items():
    parent_dir = glob_path.parent
    parquet_files = list(parent_dir.glob("*.parquet"))

    input_path_checks.append({
        "dataset": dataset_name,
        "path": str(glob_path),
        "path_type": "partitioned_glob",
        "exists": parent_dir.exists() and len(parquet_files) > 0,
        "file_count": len(parquet_files)
    })

input_path_checks_df = pd.DataFrame(input_path_checks)

display(input_path_checks_df)

,dataset,path,path_type,exists,file_count
0,traffic,pipeline_data/1.2.2.traffic_with_zones.parquet,single_file,True,1
1,bridge_tunnel,pipeline_data/1.2.1.bridge_tunnel_temporal.par...,single_file,True,1
2,taxi_zones,pipeline_data/1.1.6.nyc_taxi_zones.parquet,single_file,True,1
3,bus,pipeline_data/1.2.2.bus_speeds_spatiotemporal_...,partitioned_glob,True,39
4,subway,pipeline_data/1.2.2.subway_with_zones_parquet/...,partitioned_glob,True,13
5,tlc,pipeline_data/1.2.3.final_tables/*.parquet,partitioned_glob,True,118


### Bus issue

During the initial dataset inventory, Bus surfaced as a special case\. Unlike the other mobility datasets, the Bus output does not contain a true date field\. Before deciding how to harmonize it, we inspect the relationship between the Bus Timestamp field and the source temporal fields to determine whether the timestamp can safely be used as a calendar date\.

This query previews distinct Bus timestamp records alongside the weekday derived from the timestamp and the weekday provided by the source data\. If a single timestamp appears with multiple source weekdays, then the timestamp cannot be interpreted as a true date\-level observation\.

In [6]:
# ---------------------------------------------------------------------
# Inspect whether Bus Timestamp behaves like a true calendar date
# ---------------------------------------------------------------------

# Bus source timestamps are profile anchors, not guaranteed observed service dates.
# Compare timestamp-derived weekday against the source weekday to test that assumption directly.
bus_timestamp_diagnostic_df = duckdb.sql(f"""
    SELECT DISTINCT
        "Timestamp",
        CAST("Timestamp" AS DATE) AS timestamp_date,
        STRFTIME(CAST("Timestamp" AS DATE), '%A') AS weekday_from_timestamp,
        day_of_week AS source_day_of_week,
        "Year",
        "Month",
        hour
    FROM read_parquet('{bus_spatiotemporal_glob}')
    ORDER BY "Timestamp"
    LIMIT 50
""").df()

display(bus_timestamp_diagnostic_df)

,Timestamp,timestamp_date,weekday_from_timestamp,source_day_of_week,Year,Month,hour
0,2023-01-01 00:00:00,2023-01-01,Sunday,Sunday,2023,1,0
1,2023-01-01 00:00:00,2023-01-01,Sunday,Monday,2023,1,0
2,2023-01-01 00:00:00,2023-01-01,Sunday,Tuesday,2023,1,0
3,2023-01-01 00:00:00,2023-01-01,Sunday,Friday,2023,1,0
4,2023-01-01 00:00:00,2023-01-01,Sunday,Wednesday,2023,1,0
5,2023-01-01 00:00:00,2023-01-01,Sunday,Saturday,2023,1,0
6,2023-01-01 00:00:00,2023-01-01,Sunday,Thursday,2023,1,0
7,2023-01-01 01:00:00,2023-01-01,Sunday,Friday,2023,1,1
8,2023-01-01 01:00:00,2023-01-01,Sunday,Monday,2023,1,1
9,2023-01-01 01:00:00,2023-01-01,Sunday,Saturday,2023,1,1


Findings\. Inspection of the Bus timestamp field confirmed that individual timestamps are associated with multiple source weekdays\. For example, records sharing the same timestamp \(2023\-01\-01 00:00:00\) appear with Sunday, Monday, Tuesday, Wednesday, Thursday, Friday, and Saturday source day\-of\-week values\. This indicates that the timestamp cannot be interpreted as a true calendar\-date observation and instead functions as a temporal anchor for monthly weekday/hour profiles\.

Next, quantify how many distinct source weekdays appear for each Bus timestamp\. A true calendar timestamp should map to only one weekday; multiple weekdays per timestamp indicate that the timestamp is functioning as a month/hour anchor rather than a true observation date\.

In [7]:
# ---------------------------------------------------------------------
# Count source weekdays represented by each Bus Timestamp
# ---------------------------------------------------------------------

# A true date can only have one weekday. If each timestamp appears with several source
# weekdays, the bus table is a month/day-of-week/hour profile table rather than daily data.
bus_timestamp_weekday_check_df = duckdb.sql(f"""
    SELECT
        "Timestamp",
        CAST("Timestamp" AS DATE) AS timestamp_date,
        COUNT(DISTINCT day_of_week) AS distinct_source_weekdays
    FROM read_parquet('{bus_spatiotemporal_glob}')
    GROUP BY 1, 2
    ORDER BY distinct_source_weekdays DESC, "Timestamp"
    LIMIT 20
""").df()

display(bus_timestamp_weekday_check_df)

,Timestamp,timestamp_date,distinct_source_weekdays
0,2023-01-01 00:00:00,2023-01-01,7
1,2023-01-01 01:00:00,2023-01-01,7
2,2023-01-01 02:00:00,2023-01-01,7
3,2023-01-01 03:00:00,2023-01-01,7
4,2023-01-01 04:00:00,2023-01-01,7
5,2023-01-01 05:00:00,2023-01-01,7
6,2023-01-01 06:00:00,2023-01-01,7
7,2023-01-01 07:00:00,2023-01-01,7
8,2023-01-01 08:00:00,2023-01-01,7
9,2023-01-01 09:00:00,2023-01-01,7


Findings\. Each Bus timestamp is associated with all seven source weekdays rather than a single weekday\. This behavior would be impossible if the timestamp represented a true calendar\-date observation\. Instead, the timestamp appears to serve as a temporal anchor for monthly weekday/hour profiles, confirming that the dataset is organized at a month × day\-of\-week × hour level rather than a true date level\.

Finally, summarize the Bus timestamp structure\. This confirms whether the dataset is organized around monthly day\-of\-week/hour profiles rather than true calendar\-date observations\.

In [8]:
bus_timestamp_summary_df = duckdb.sql(f"""
    SELECT
        MIN("Timestamp") AS min_timestamp,
        MAX("Timestamp") AS max_timestamp,
        COUNT(DISTINCT "Timestamp") AS distinct_timestamps,
        COUNT(DISTINCT CAST("Timestamp" AS DATE)) AS distinct_timestamp_dates,
        COUNT(*) AS total_rows,
        COUNT(DISTINCT day_of_week) AS distinct_source_weekdays,
        COUNT(DISTINCT hour) AS distinct_hours
    FROM read_parquet('{bus_spatiotemporal_glob}')
""").df()

display(bus_timestamp_summary_df)

,min_timestamp,max_timestamp,distinct_timestamps,distinct_timestamp_dates,total_rows,distinct_source_weekdays,distinct_hours
0,2023-01-01,2026-03-01 23:00:00,936,39,18937024,7,24


Findings\. Additional validation confirmed that the dataset contains only 936 distinct timestamps and 39 distinct dates across more than 18\.9 million observations\. This aligns with the dataset's known structure of monthly weekday/hour profiles rather than individual daily observations\. The presence of all seven weekdays and all 24 hours across a relatively small set of timestamps further confirms that the timestamp field should not be used directly for date\-level harmonization\.

### Section Summary

The Bus dataset does not contain true date\-level observations\. Instead, each timestamp appears with multiple source weekdays, indicating that the timestamp functions as a month/hour anchor while the source data separately defines the day\-of\-week profile\. This means Bus speeds represent average conditions by year, month, day of week, hour, and route segment rather than observed speeds for a specific calendar date\. To support the project's Taxi Zone × Date × Temporal Bucket integration grain, Bus will need to be expanded across matching calendar dates during final harmonization\.

### Get a Bird's Eye View of our Modality Multimodal Datasets

With the Bus exception documented, we now create a lightweight inventory of the standardized datasets\. For datasets with true date fields, we summarize date coverage directly\. For Bus, we summarize year/month coverage and flag that date\-level expansion is required before final harmonized output can be written\.

In [9]:
dataset_inventory_queries = [
    {
        "dataset": "traffic",
        "path": str(traffic_spatiotemporal_path),
        "date_field": "date",
        "spatial_field": "location_id",
        "measure_field": "traffic_volume",
        "requires_date_expansion": False
    },
    {
        "dataset": "bus",
        "path": str(bus_spatiotemporal_glob),
        "date_field": None,
        "spatial_field": "bus_taxi_zone_id",
        "measure_field": '"Average Road Speed"',
        "requires_date_expansion": True
    },
    {
        "dataset": "subway",
        "path": str(subway_spatiotemporal_glob),
        "date_field": "transit_timestamp",
        "spatial_field": "taxi_zone_id",
        "measure_field": "ridership",
        "requires_date_expansion": False
    },
    {
        "dataset": "tlc",
        "path": str(tlc_mobility_glob),
        "date_field": "date",
        "spatial_field": "pickup_zone_id",
        "measure_field": "trip_count",
        "requires_date_expansion": False
    },
    {
        "dataset": "bridge_tunnel",
        "path": str(bridge_tunnel_temporal_path),
        "date_field": "date",
        "spatial_field": "facility_id",
        "measure_field": "traffic_count",
        "requires_date_expansion": False
    },
]

In [10]:
# ---------------------------------------------------------------------
# Generate or Load Harmonization Inventory Manifest
# ---------------------------------------------------------------------

harmonization_inventory_manifest_path = (
    PIPELINE_DATA_DIR /
    "qa" /
    "harmonization_inventory_manifest.parquet"
)

if (
    REBUILD_HARMONIZATION_INVENTORY
    or not harmonization_inventory_manifest_path.exists()
):

    print("Generating harmonization inventory manifest...")

    dataset_inventory_rows = []

    for dataset_info in dataset_inventory_queries:
        dataset_name = dataset_info["dataset"]
        path = dataset_info["path"]
        date_field = dataset_info["date_field"]
        spatial_field = dataset_info["spatial_field"]
        measure_field = dataset_info["measure_field"]
        requires_date_expansion = dataset_info["requires_date_expansion"]

        if date_field is None:
            inventory_query = f"""
                SELECT
                    '{dataset_name}' AS dataset,
                    COUNT(*) AS row_count,
                    NULL AS min_date,
                    NULL AS max_date,
                    MIN("Year") AS min_year,
                    MAX("Year") AS max_year,
                    COUNT(DISTINCT "Month") AS distinct_months,
                    COUNT(DISTINCT day_of_week) AS distinct_days_of_week,
                    COUNT(DISTINCT temporal_bucket) AS temporal_bucket_count,
                    COUNT(DISTINCT pre_post_cp) AS pre_post_cp_count,
                    COUNT(DISTINCT {spatial_field}) AS distinct_spatial_values,
                    COUNT(*) FILTER (
                        WHERE {measure_field} IS NULL
                    ) AS missing_measure_count,
                    {str(requires_date_expansion).lower()}
                        AS requires_date_expansion
                FROM read_parquet('{path}')
            """
        else:
            inventory_query = f"""
                SELECT
                    '{dataset_name}' AS dataset,
                    COUNT(*) AS row_count,
                    MIN({date_field}) AS min_date,
                    MAX({date_field}) AS max_date,
                    NULL AS min_year,
                    NULL AS max_year,
                    NULL AS distinct_months,
                    NULL AS distinct_days_of_week,
                    COUNT(DISTINCT temporal_bucket)
                        AS temporal_bucket_count,
                    COUNT(DISTINCT pre_post_cp)
                        AS pre_post_cp_count,
                    COUNT(DISTINCT {spatial_field})
                        AS distinct_spatial_values,
                    COUNT(*) FILTER (
                        WHERE {measure_field} IS NULL
                    ) AS missing_measure_count,
                    {str(requires_date_expansion).lower()}
                        AS requires_date_expansion
                FROM read_parquet('{path}')
            """

        dataset_inventory_rows.append(
            duckdb.sql(inventory_query).df()
        )

    dataset_inventory_df = pd.concat(
        dataset_inventory_rows,
        ignore_index=True
    )

    harmonization_inventory_manifest_path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    dataset_inventory_df.to_parquet(
        harmonization_inventory_manifest_path,
        index=False
    )

    print(
        f"Saved manifest to: "
        f"{harmonization_inventory_manifest_path}"
    )

else:

    print("Loading harmonization inventory manifest...")

    dataset_inventory_df = pd.read_parquet(
        harmonization_inventory_manifest_path
    )

display(dataset_inventory_df)

Loading harmonization inventory manifest...


,dataset,row_count,min_date,max_date,min_year,max_year,distinct_months,distinct_days_of_week,temporal_bucket_count,pre_post_cp_count,distinct_spatial_values,missing_measure_count,requires_date_expansion
0,traffic,270011,2023-01-06,2026-02-03 00:00:00,<NA>,<NA>,<NA>,<NA>,10,2,138,0,False
1,bus,18937024,NaT,NaT,2023,2026,12,7,10,2,252,0,True
2,subway,88319707,2023-01-01,2026-03-30 07:00:00,<NA>,<NA>,<NA>,<NA>,10,2,156,0,False
3,tlc,327863727,2001-01-01,2026-06-26 00:00:00,<NA>,<NA>,<NA>,<NA>,10,2,263,0,False
4,bridge_tunnel,5941375,2023-01-01,2026-05-12 00:00:00,<NA>,<NA>,<NA>,<NA>,10,2,10,0,False


Findings\. The harmonized dataset inventory confirmed that all five mobility datasets are available for downstream integration and contain the expected temporal standardization fields introduced during earlier processing stages\. However, several important harmonization considerations remain\. The Bus dataset is the only modality that does not currently contain true date\-level observations and will require temporal expansion before it can participate in the project's Taxi Zone × Date × Temporal Bucket integration framework\. The inventory also confirmed that temporal coverage differs across modalities, with Traffic spanning January 2023 through February 2026, Subway through March 2026, Bridge & Tunnel through May 2026, and TLC through June 2026\. Finally, TLC contains observations extending well beyond the project's intended analysis period, indicating that study\-period filtering will be required during final harmonization\. No missing mobility measures were identified in any dataset\.

### Traffic data investigation

Hmm, let's see what's going on in the Traffic dataset\. During the inventory review, we noticed that Traffic begins several days later than the other mobility datasets and ends earlier than several modalities\. Before deciding how to align study\-period coverage across datasets, it is worth understanding whether the Traffic data represents continuous daily observations or a more periodic traffic\-count collection program\.

In [11]:
traffic_date_coverage_df = duckdb.sql(f"""
    SELECT
        COUNT(DISTINCT date) AS distinct_dates,
        MIN(date) AS min_date,
        MAX(date) AS max_date
    FROM read_parquet('{traffic_spatiotemporal_path}')
""").df()

display(traffic_date_coverage_df)

,distinct_dates,min_date,max_date
0,714,2023-01-06,2026-02-03


Findings\. The Traffic dataset contains observations spanning January 2023 through February 2026 but only includes 714 distinct dates across the study period\. Given that the calendar window contains more than 1,100 days, this indicates that Traffic does not provide continuous daily observations\. Instead, the dataset appears to consist of periodic traffic\-count collections conducted on selected dates throughout the study period\.

In [12]:
traffic_gap_df = duckdb.sql(f"""
WITH dates AS (
    SELECT DISTINCT date
    FROM read_parquet('{traffic_spatiotemporal_path}')
),
date_gaps AS (
    SELECT
        date,
        LAG(date) OVER (ORDER BY date) AS previous_date
    FROM dates
)
SELECT
    date,
    previous_date,
    date - previous_date AS gap_days
FROM date_gaps
WHERE previous_date IS NOT NULL
ORDER BY gap_days DESC
LIMIT 20
""").df()

display(traffic_gap_df)

,date,previous_date,gap_days
0,2024-09-07,2024-06-24,75
1,2025-09-06,2025-07-19,49
2,2024-01-06,2023-12-10,27
3,2025-01-06,2024-12-11,26
4,2023-09-06,2023-08-12,25
5,2023-07-29,2023-07-07,22
6,2023-02-27,2023-02-06,21
7,2026-01-03,2025-12-14,20
8,2024-10-08,2024-09-21,17
9,2025-10-04,2025-09-19,15


Findings\. The date\-gap analysis confirmed the presence of substantial gaps between traffic collection periods, including intervals of 75, 49, 27, and 25 days between observed dates\. These gaps are too large to be explained by occasional missing observations and instead suggest that the underlying traffic program consists of periodic roadway count studies rather than continuously monitored traffic sensors\. Because these gaps reflect the structure of the source data rather than missing records, no temporal interpolation will be performed\. Traffic observations will be retained only for dates that were actually collected by the source program\.

Finally, inspect each dataset schema using DuckDB's DESCRIBE output\. This provides a lightweight view of column names and data types so we can identify any obvious schema mismatches before applying harmonization fixes\.

### Schema Expansion

In [13]:
schema_rows = []

for dataset_info in dataset_inventory_queries:
    dataset_name = dataset_info["dataset"]
    path = dataset_info["path"]

    schema_df = duckdb.sql(f"""
        DESCRIBE
        SELECT *
        FROM read_parquet('{path}')
    """).df()

    schema_df.insert(0, "dataset", dataset_name)
    schema_rows.append(schema_df)

harmonized_schema_df = pd.concat(
    schema_rows,
    ignore_index=True
)

display(harmonized_schema_df)

,dataset,column_name,column_type,null,key,default,extra
0,traffic,segment_id,BIGINT,YES,None,None,None
1,traffic,borough_left,VARCHAR,YES,None,None,None
2,traffic,street_name,VARCHAR,YES,None,None,None
3,traffic,from_street,VARCHAR,YES,None,None,None
4,traffic,to_street,VARCHAR,YES,None,None,None
...,...,...,...,...,...,...,...
100,bridge_tunnel,year,INTEGER,YES,None,None,None
101,bridge_tunnel,month,INTEGER,YES,None,None,None
102,bridge_tunnel,day_of_week,VARCHAR,YES,None,None,None
103,bridge_tunnel,temporal_bucket,VARCHAR,YES,None,None,None


Findings\. Schema inspection confirmed that all mobility datasets contain the standardized temporal fields required for downstream integration, including temporal\_bucket and pre\_post\_cp\. However, several modality\-specific naming conventions remain\. Traffic currently uses location\_id as its geographic identifier, Bus uses bus\_taxi\_zone\_id, Subway already uses the project's preferred taxi\_zone\_id convention, TLC uses separate pickup\_zone\_id and dropoff\_zone\_id fields, and Bridge & Tunnel remains organized around facility\_id rather than Taxi Zones\.

The review also highlighted several remaining harmonization tasks\. Bus retains duplicate temporal fields \(Year/year\_1, Month/month\_1, Day of Week/day\_of\_week, and Hour of Day/hour\) that should be consolidated during final harmonization\. In addition, Bus continues to use source\-specific measure names such as Average Road Speed and Average Travel Time, which should be standardized to integration\-friendly field names\. Traffic and Bridge & Tunnel retain modality\-specific geographic identifiers \(location\_id and facility\_id, respectively\), while TLC continues to use separate pickup and dropoff Taxi Zone identifiers\. Overall, the schemas appear stable and suitable for downstream harmonization, with the Bus temporal expansion, a small number of field standardizations, and TLC spatial reference updates representing the primary remaining integration tasks\.

## 1\.2\.4\.2 Harmonize Bus Temporal Representation

The previous section confirmed that the Bus dataset is organized as monthly weekday/hour profiles rather than true date\-level observations\. Because the project's target integration grain is Taxi Zone × Date × Temporal Bucket, the Bus dataset cannot participate directly in downstream multimodal integration in its current form\. In this section, the monthly weekday/hour profiles will be expanded across the matching calendar dates within each month, creating a date\-level representation that aligns with the project's common temporal framework\. During this process, redundant source fields will be removed, integration\-ready field names will be standardized, and a final harmonized Bus dataset will be written for downstream analysis\.

### Proposed Bus Harmonization Plan

The Bus dataset currently contains a mixture of source fields, intermediate harmonization fields, and spatial assignment fields\. Before creating the final harmonized output, we will simplify the schema, standardize naming conventions, and align the dataset to the project's common Taxi Zone × Date × Temporal Bucket integration framework\.

### Fields to Retain

The following fields will be retained because they provide route\-level context, support future diagnostics, or represent core mobility measures:

Route ID
Direction
Borough
Route Type
Stop Order
Timepoint Stop ID
Timepoint Stop Name
Next Timepoint Stop ID
Next Timepoint Stop Name
Road Distance
bus\_taxi\_zone\_assignment\_method

### Fields to Standardize

The following fields will be renamed to align with project\-wide naming conventions:

Average Road Speed → avg\_bus\_speed, bus\_speed\_stddev
Average Travel Time → avg\_bus\_travel\_time, bus\_travel\_time\_stddev
Bus Trip Count → bus\_trip\_count
bus\_taxi\_zone\_id → taxi\_zone\_id

### Temporal Harmonization Changes

The source dataset is organized as monthly weekday/hour profiles rather than true date\-level observations\. To support downstream multimodal integration, monthly weekday/hour profiles will be expanded across the matching calendar dates within each month\.

The final harmonized temporal fields will be:

date
year\_1 → year
month\_1 → month
day\_of\_week
hour
temporal\_bucket
pre\_post\_cp

The pre\_post\_cp field will be recalculated using the generated calendar dates\.

### Fields to Remove

The following fields will be removed because they are either redundant or were only required during earlier processing stages:

Timestamp
Year
Month
Day of Week
Hour of Day
Timepoint Stop Latitude
Timepoint Stop Longitude
Next Timepoint Stop Latitude
Next Timepoint Stop Longitude
Timepoint Stop Georeference
Next Timepoint Stop Georeference

### Proposed Final Harmonized Bus Schema

date
year
month
day\_of\_week
temporal\_bucket
pre\_post\_cp
taxi\_zone\_id
bus\_taxi\_zone\_assignment\_method
avg\_bus\_speed
bus\_speed\_stddev
avg\_bus\_travel\_time
bus\_travel\_time\_stddev
bus\_trip\_count
bus\_segment\_observation\_count

This schema preserves route\-level analytical value while aligning the Bus dataset with the common temporal and spatial framework used by the other mobility datasets\.

📝 Note: In addition to preserving average speed and average travel time, the harmonized Bus mobility table will capture within\-bucket variability using standard deviation metrics for both measures\. These fields quantify how consistently roadway segments within a Taxi Zone are performing during a given temporal bucket and may provide useful signals for anomaly detection, clustering, forecasting, and congestion\-pricing analysis\. Because standard deviation is undefined when only a single segment observation contributes to a group, singleton observations are assigned a value of 0 to avoid introducing unnecessary missing values into downstream modeling workflows\.

### Build calendar expansion

First, create a compact calendar table for the study period\. This table gives us every valid calendar date, along with its year, month, and day of week, so we can map Bus monthly weekday profiles onto the matching dates\.

In [14]:
# ---------------------------------------------------------------------
# 1.2.4.2 Harmonize Bus Temporal Representation
# ---------------------------------------------------------------------

bus_expansion_calendar_df = pd.DataFrame({
    "date": pd.date_range(
        start=STUDY_START_DATE,
        end=STUDY_END_DATE,
        freq="D"
    )
})

bus_expansion_calendar_df["year"] = (
    bus_expansion_calendar_df["date"].dt.year
)

bus_expansion_calendar_df["month"] = (
    bus_expansion_calendar_df["date"].dt.month
)

bus_expansion_calendar_df["day_of_week"] = (
    bus_expansion_calendar_df["date"].dt.day_name()
)

bus_expansion_calendar_df.head()

,date,year,month,day_of_week
0,2023-01-01,2023,1,Sunday
1,2023-01-02,2023,1,Monday
2,2023-01-03,2023,1,Tuesday
3,2023-01-04,2023,1,Wednesday
4,2023-01-05,2023,1,Thursday


Next, register the calendar as a DuckDB view\. This allows us to join the Bus monthly weekday/hour profiles to matching calendar dates without loading the full Bus dataset into Pandas\.

In [15]:
duckdb.register(
    "bus_expansion_calendar",
    bus_expansion_calendar_df
)

### Estimate Expansion Size

Before writing the final output, let's run a small preview to confirm that Bus records join to the expected calendar dates\. A single monthly weekday profile should expand to all matching dates for that weekday within the same month\.

In [16]:
bus_expansion_preview_df = duckdb.sql(f"""
    SELECT
        b."Year",
        b."Month",
        b.day_of_week,
        b.hour,
        c.date,
        b.bus_taxi_zone_id,
        b."Average Road Speed",
        b."Average Travel Time",
        b."Bus Trip Count"
    FROM read_parquet('{bus_spatiotemporal_glob}') AS b
    INNER JOIN bus_expansion_calendar AS c
        ON b.year_1 = c.year
        AND b.month_1 = c.month
        AND b.day_of_week = c.day_of_week
    WHERE
        b.year_1 = 2023
        AND b.month_1 = 1
        AND b.day_of_week = 'Sunday'
        AND b.hour = 3
    LIMIT 25
""").df()

display(bus_expansion_preview_df)

,Year,Month,day_of_week,hour,date,bus_taxi_zone_id,Average Road Speed,Average Travel Time,Bus Trip Count
0,2023,1,Sunday,3,2023-01-29,130.0,10.826714,5.608350,2
1,2023,1,Sunday,3,2023-01-29,37.0,10.400779,6.703344,5
2,2023,1,Sunday,3,2023-01-29,65.0,7.441222,3.233340,3
3,2023,1,Sunday,3,2023-01-29,47.0,10.198234,6.112824,13
4,2023,1,Sunday,3,2023-01-29,130.0,9.177010,6.609996,5
5,2023,1,Sunday,3,2023-01-29,109.0,21.032786,8.230008,5
6,2023,1,Sunday,3,2023-01-29,131.0,15.444127,3.309996,5
7,2023,1,Sunday,3,2023-01-29,227.0,14.268115,2.001666,10
8,2023,1,Sunday,3,2023-01-29,7.0,10.590543,12.203340,5
9,2023,1,Sunday,3,2023-01-29,155.0,19.304871,1.563336,5


Findings: The expansion preview confirms that the Bus monthly weekday/hour profiles can be successfully mapped to calendar dates within the matching month\. For example, January 2023 Sunday 3 AM observations were expanded across all matching Sundays in January \(January 8, 15, 22, and 29\)\. The underlying speed, travel time, and trip\-count metrics remain unchanged during expansion, while the newly generated date field provides the date\-level representation required for downstream integration\. This confirms that the proposed expansion approach preserves the original monthly profile values while creating a valid date\-level temporal structure\.

Now estimate the size of the expanded Bus table before writing it\. This gives us a sanity check on how many rows the final date\-level Bus mobility table will contain\.

In [17]:

bus_expanded_row_count_estimate_df = duckdb.sql(f"""
    SELECT
        COUNT(*) AS estimated_expanded_rows
    FROM read_parquet('{bus_spatiotemporal_glob}') AS b
    INNER JOIN bus_expansion_calendar AS c
        ON b.year_1 = c.year
        AND b.month_1 = c.month
        AND b.day_of_week = c.day_of_week
""").df()

display(bus_expanded_row_count_estimate_df)

,estimated_expanded_rows
0,82363372


Findings: The estimated expansion would increase the Bus dataset from approximately 18\.9 million monthly weekday/hour observations to approximately 78\.2 million date\-level observations\. While technically feasible, this expansion would substantially increase storage and processing requirements without improving the analytical grain required for downstream integration\. As a result, the Bus harmonization process will aggregate observations directly to the Taxi Zone × Date × Temporal Bucket level immediately after expansion, avoiding the creation of a large intermediate dataset while still producing the integration\-ready mobility table required for multimodal analysis\.

### Build Harmonized Bus Dataset

Next, create the final harmonized Bus mobility table\. Rather than writing expanded route\-segment\-level records, we aggregate directly to Taxi Zone × Date × Temporal Bucket, which is the grain needed for downstream integration\.

In [18]:
if REBUILD_HARMONIZED_BUS or not harmonized_bus_path.exists():

    print("Building harmonized Bus mobility table...")

    duckdb.sql(f"""
        COPY (
            SELECT
                c.date::DATE AS date,
                EXTRACT(YEAR FROM c.date)::INTEGER AS year,
                EXTRACT(MONTH FROM c.date)::INTEGER AS month,
                STRFTIME(c.date, '%A') AS day_of_week,
                b.temporal_bucket,
                CASE
                    WHEN c.date >= DATE '2025-01-05'
                    THEN 'post_cp'
                    ELSE 'pre_cp'
                END AS pre_post_cp,
                b.bus_taxi_zone_id::INTEGER AS taxi_zone_id,

                AVG(b."Average Road Speed") AS avg_bus_speed,
                AVG(b."Average Travel Time") AS avg_bus_travel_time,

                COALESCE(
                    STDDEV_SAMP(b."Average Road Speed"),
                    0
                ) AS bus_speed_stddev,

                COALESCE(
                    STDDEV_SAMP(b."Average Travel Time"),
                    0
                ) AS bus_travel_time_stddev,
                SUM(b."Bus Trip Count") AS bus_trip_count,
                COUNT(*) AS bus_segment_observation_count,
                COUNT(DISTINCT b.bus_taxi_zone_assignment_method) AS bus_assignment_method_count

            FROM read_parquet('{bus_spatiotemporal_glob}') AS b
            INNER JOIN bus_expansion_calendar AS c
                ON b.year_1 = c.year
                AND b.month_1 = c.month
                AND b.day_of_week = c.day_of_week

            WHERE
                b.bus_taxi_zone_id IS NOT NULL

            GROUP BY
                c.date,
                b.temporal_bucket,
                b.bus_taxi_zone_id
        )
        TO '{harmonized_bus_path}'
        (FORMAT PARQUET, COMPRESSION ZSTD)
    """)

    print(f"Saved harmonized Bus table to: {harmonized_bus_path}")

else:
    print(f"Harmonized Bus table already exists: {harmonized_bus_path}")

Harmonized Bus table already exists: pipeline_data/1.2.4.harmonized_mobility_datasets/bus_mobility_table.parquet


### QA Harmonized Bus Dataset

Finally, inspect the saved Bus output to confirm that it now has the expected date\-level structure and standardized field names\.

In [19]:
harmonized_bus_check_df = duckdb.sql(f"""
    SELECT
        COUNT(*) AS row_count,
        MIN(date) AS min_date,
        MAX(date) AS max_date,
        COUNT(DISTINCT taxi_zone_id) AS distinct_taxi_zones,
        COUNT(DISTINCT temporal_bucket) AS distinct_temporal_buckets,
        COUNT(DISTINCT pre_post_cp) AS distinct_pre_post_values,
        COUNT(*) FILTER (WHERE avg_bus_speed IS NULL) AS missing_avg_bus_speed
    FROM read_parquet('{harmonized_bus_path}')
""").df()

display(harmonized_bus_check_df)

,row_count,min_date,max_date,distinct_taxi_zones,distinct_temporal_buckets,distinct_pre_post_values,missing_avg_bus_speed
0,1476478,2023-01-01,2026-03-31,252,10,2,0


Findings\. The final harmonized Bus mobility table now contains approximately 1\.4 million Taxi Zone × Date × Temporal Bucket observations spanning January 2023 through February 2026\. The aggregation removes raw hourly rows and aligns Bus with the project’s canonical analytical grain, while preserving coverage across 252 mapped Taxi Zones, all 10 standardized temporal buckets, and both pre\- and post\-congestion\-pricing periods\. No missing average\-speed values were identified\. The table now includes core Bus mobility measures plus diagnostic fields, including segment observation counts, assignment\-method counts, and bus speed variability for future anomaly detection and clustering\.

Preview a few rows from the final Bus table so we can confirm that the output is now organized at the expected integration grain\.

In [20]:
harmonized_bus_preview_df = duckdb.sql(f"""
    SELECT *
    FROM read_parquet('{harmonized_bus_path}')
    ORDER BY date, taxi_zone_id, temporal_bucket
    LIMIT 20
""").df()

display(harmonized_bus_preview_df)

,date,year,month,day_of_week,temporal_bucket,pre_post_cp,taxi_zone_id,avg_bus_speed,avg_bus_travel_time,bus_speed_stddev,bus_travel_time_stddev,bus_trip_count,bus_segment_observation_count,bus_assignment_method_count
0,2023-01-01,2023,1,Sunday,weekend_am_peak,pre_cp,3,11.653357,7.364015,3.648081,2.781329,588.0,50,1
1,2023-01-01,2023,1,Sunday,weekend_evening,pre_cp,3,11.382051,7.796905,2.556591,3.023619,551.0,50,1
2,2023-01-01,2023,1,Sunday,weekend_midday,pre_cp,3,10.012420,8.737292,2.243434,3.010996,1033.0,78,1
3,2023-01-01,2023,1,Sunday,weekend_overnight,pre_cp,3,13.056119,7.339687,4.704994,2.856047,152.0,18,1
4,2023-01-01,2023,1,Sunday,weekend_pm_peak,pre_cp,3,9.728666,8.993071,1.981879,3.281412,670.0,52,1
5,2023-01-01,2023,1,Sunday,weekend_am_peak,pre_cp,4,8.279115,4.612576,1.307143,1.430721,82.0,8,1
6,2023-01-01,2023,1,Sunday,weekend_evening,pre_cp,4,7.591659,4.995253,0.518575,1.254371,116.0,11,1
7,2023-01-01,2023,1,Sunday,weekend_midday,pre_cp,4,6.164488,6.379933,0.635109,1.889375,204.0,18,1
8,2023-01-01,2023,1,Sunday,weekend_overnight,pre_cp,4,8.210226,3.855556,0.286053,0.586626,25.0,3,1
9,2023-01-01,2023,1,Sunday,weekend_pm_peak,pre_cp,4,5.979422,6.543762,0.502382,1.846901,134.0,12,1


Findings\. The preview confirms that the final harmonized Bus mobility table has been successfully transformed into the project's integration\-ready structure\. Observations are now organized at the Taxi Zone × Date × Temporal Bucket level while preserving key mobility measures, including average bus speed, average travel time, and trip volume\. The generated date field, standardized temporal attributes, congestion\-pricing indicators, and Taxi Zone identifiers are all present and align with the project's common analytical framework\.

Before moving on, we run one final grain validation check to confirm that the harmonized Bus table contains exactly one row per Taxi Zone × Date × Temporal Bucket combination\. This is the agreed aggregation dimension for the final mobility table, so any duplicate keys would indicate that source\-level detail was still leaking into the harmonized output\.

In [21]:
# Validate Bus mobility table grain

bus_grain_validation_df = duckdb.sql(f"""
    SELECT
        COUNT(*) AS row_count,
        COUNT(DISTINCT taxi_zone_id || '|' || date || '|' || temporal_bucket) AS distinct_taxi_zone_date_bucket_keys,
        COUNT(*) - COUNT(DISTINCT taxi_zone_id || '|' || date || '|' || temporal_bucket) AS duplicate_key_count
    FROM read_parquet('{harmonized_bus_path}')
""").df()

print("Bus Mobility Table Grain Validation:")
display(bus_grain_validation_df)

bus_duplicate_grain_df = duckdb.sql(f"""
    SELECT
        taxi_zone_id,
        date,
        temporal_bucket,
        COUNT(*) AS row_count
    FROM read_parquet('{harmonized_bus_path}')
    GROUP BY 1,2,3
    HAVING COUNT(*) > 1
    ORDER BY row_count DESC
    LIMIT 20
""").df()

print("Duplicate Taxi Zone × Date × Temporal Bucket Records:")
display(bus_duplicate_grain_df)

Bus Mobility Table Grain Validation:


,row_count,distinct_taxi_zone_date_bucket_keys,duplicate_key_count
0,1476478,1476478,0


Duplicate Taxi Zone × Date × Temporal Bucket Records:


,taxi_zone_id,date,temporal_bucket,row_count


Findings\. The grain validation confirms that the harmonized Bus table contains exactly one record per Taxi Zone × Date × Temporal Bucket combination, with no duplicate keys identified\. This verifies that hourly and route\-segment\-level observations have been successfully consolidated into the project's canonical mobility\-state framework\. The retained diagnostic fields, including bus\_segment\_observation\_count and bus\_assignment\_method\_count, preserve visibility into the underlying observations contributing to each aggregated record while maintaining a clean integration\-ready schema\. The Bus mobility table is now fully aligned with the project's common multimodal analytical grain and ready for downstream feature engineering, anomaly detection, clustering, forecasting, and congestion\-pricing analysis\.

### Section Summary

In this section, we resolved the final major harmonization challenge within the Bus dataset\. By expanding monthly weekday/hour profiles across matching calendar dates and immediately aggregating to the Taxi Zone × Date × Temporal Bucket level, we transformed the dataset into an integration\-ready mobility table while avoiding the creation of a much larger intermediate dataset\. The final harmonized Bus table now uses standardized field names, aligns with the project's common temporal framework, preserves full Taxi Zone coverage, and provides a consistent representation of bus mobility activity that can be directly integrated with the other mobility modalities\.

In [22]:
# Clean up Bus QA and preview DataFrames

for var_name in [
    "bus_expansion_calendar_df",
    "bus_expansion_preview_df",
    "bus_expanded_row_count_estimate_df",
    "harmonized_bus_check_df",
    "harmonized_bus_preview_df",
    "bus_grain_validation_df",
    "bus_duplicate_grain_df",
]:
    if var_name in globals():
        del globals()[var_name]

gc.collect()

10

## 1\.2\.4\.3 Finalize Taxi Zone Reference Layer

Previous TLC harmonization work identified several location identifiers that appear within the TLC mobility tables but do not currently resolve against the project's Taxi Zone reference layer\. Specifically, LocationIDs 57 and 105 appear as valid observed pickup and dropoff locations, while LocationIDs 264 and 265 represent TLC's special Unknown location codes\. In this section, we finalize the Taxi Zone reference layer by adding these missing identifiers so that all observed TLC location codes resolve cleanly during downstream mobility integration\.

### Unrelated Newark Sanity Check

In [23]:
# ---------------------------------------------------------------------
# Quick sanity check: LocationID 1 / Newark Airport across modalities
# ---------------------------------------------------------------------

location_1_checks = []

# Traffic
location_1_checks.append(
    duckdb.sql(f"""
        SELECT
            'traffic' AS dataset,
            'location_id' AS location_field,
            COUNT(*) AS observation_count,
            NULL AS trip_count
        FROM read_parquet('{traffic_spatiotemporal_path}')
        WHERE location_id = 1
    """).df()
)

# Bus
location_1_checks.append(
    duckdb.sql(f"""
        SELECT
            'bus' AS dataset,
            'bus_taxi_zone_id' AS location_field,
            COUNT(*) AS observation_count,
            NULL AS trip_count
        FROM read_parquet('{bus_spatiotemporal_glob}')
        WHERE bus_taxi_zone_id = 1
    """).df()
)

# Subway
location_1_checks.append(
    duckdb.sql(f"""
        SELECT
            'subway' AS dataset,
            'taxi_zone_id' AS location_field,
            COUNT(*) AS observation_count,
            NULL AS trip_count
        FROM read_parquet('{subway_spatiotemporal_glob}')
        WHERE taxi_zone_id = 1
    """).df()
)

# TLC pickups
location_1_checks.append(
    duckdb.sql(f"""
        SELECT
            'tlc_pickup' AS dataset,
            'pickup_zone_id' AS location_field,
            COUNT(*) AS observation_count,
            SUM(trip_count) AS trip_count
        FROM read_parquet('{tlc_mobility_glob}')
        WHERE pickup_zone_id = 1
    """).df()
)

# TLC dropoffs
location_1_checks.append(
    duckdb.sql(f"""
        SELECT
            'tlc_dropoff' AS dataset,
            'dropoff_zone_id' AS location_field,
            COUNT(*) AS observation_count,
            SUM(trip_count) AS trip_count
        FROM read_parquet('{tlc_mobility_glob}')
        WHERE dropoff_zone_id = 1
    """).df()
)

location_1_check_df = pd.concat(
    location_1_checks,
    ignore_index=True
)

display(location_1_check_df)

/tmp/ipykernel_333/854094052.py:72: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  location_1_check_df = pd.concat(


,dataset,location_field,observation_count,trip_count
0,traffic,location_id,0,NaN
1,bus,bus_taxi_zone_id,0,NaN
2,subway,taxi_zone_id,0,NaN
3,tlc_pickup,pickup_zone_id,2136,2292.0
4,tlc_dropoff,dropoff_zone_id,1975659,5879274.0


Findings: LocationID 1, which represents Newark Airport, appears only in the TLC mobility tables and does not appear in the Traffic, Bus, or Subway spatial assignments\. This is consistent with expectations because Newark Airport is a valid TLC destination but does not correspond to NYC\-based roadway sensors, bus route segments, or subway station locations in the other mobility datasets\.

### Load Reference Layer and Locate Problematic IDs

Load the current Taxi Zone reference layer and confirm the schema before making changes\. This helps us make sure any added records match the existing field structure\.

In [24]:
# ---------------------------------------------------------------------
# 1.2.4.3 Finalize Taxi Zone Reference Layer
# ---------------------------------------------------------------------

taxi_zones_df = pd.read_parquet(
    taxi_zones_path
)

print("Current Taxi Zone reference shape:", taxi_zones_df.shape)

display(taxi_zones_df.head())
display(taxi_zones_df.dtypes.reset_index().rename(columns={"index": "field", 0: "dtype"}))

Current Taxi Zone reference shape: (263, 6)


,location_id,zone,borough,shape_length,shape_area,geometry
0,1,Newark Airport,EWR,0.116357,0.000782,b'\x01\x06\x00\x00\x00\x01\x00\x00\x00\x01\x03...
1,2,Jamaica Bay,Queens,0.433470,0.004866,b'\x01\x06\x00\x00\x00!\x00\x00\x00\x01\x03\x0...
2,3,Allerton/Pelham Gardens,Bronx,0.084341,0.000314,b'\x01\x06\x00\x00\x00\x01\x00\x00\x00\x01\x03...
3,4,Alphabet City,Manhattan,0.043567,0.000112,b'\x01\x06\x00\x00\x00\x01\x00\x00\x00\x01\x03...
4,5,Arden Heights,Staten Island,0.092146,0.000498,b'\x01\x06\x00\x00\x00\x01\x00\x00\x00\x01\x03...


,field,dtype
0,location_id,int64
1,zone,object
2,borough,object
3,shape_length,float64
4,shape_area,float64
5,geometry,object


Next, let's confirm again whether the known TLC location identifiers already exist in the current Taxi Zone layer\. This gives us a clear before\-state before adding the missing records\.

In [25]:
tlc_location_ids_to_resolve = [57, 105, 264, 265]

existing_tlc_location_check_df = (
    taxi_zones_df[
        taxi_zones_df["location_id"].isin(tlc_location_ids_to_resolve)
    ]
    .sort_values("location_id")
)

display(existing_tlc_location_check_df)

,location_id,zone,borough,shape_length,shape_area,geometry


Findings: The existing Taxi Zone reference layer does not contain any of the TLC location identifiers requiring resolution\. LocationIDs 57, 105, 264, and 265 are all absent from the current spatial reference layer, confirming the need to add these records before performing downstream multimodal integration and spatial joins\.

Now let's define the missing Taxi Zone reference records\. LocationIDs 57 and 105 are added as observed TLC component locations, while 264 and 265 are added as explicit Unknown locations so downstream joins can remain lossless\.

In [26]:
missing_taxi_zone_records = pd.DataFrame([
    {
        "location_id": 57,
        "borough": "Queens",
        "zone": "Corona",
        "service_zone": "Boro Zone",
    },
    {
        "location_id": 105,
        "borough": "Manhattan",
        "zone": "Governor's Island/Ellis Island/Liberty Island",
        "service_zone": "Yellow Zone",
    },
    {
        "location_id": 264,
        "borough": "Unknown",
        "zone": "Unknown",
        "service_zone": "Unknown",
    },
    {
        "location_id": 265,
        "borough": "Unknown",
        "zone": "Unknown",
        "service_zone": "Unknown",
    },
])

display(missing_taxi_zone_records)

,location_id,borough,zone,service_zone
0,57,Queens,Corona,Boro Zone
1,105,Manhattan,Governor's Island/Ellis Island/Liberty Island,Yellow Zone
2,264,Unknown,Unknown,Unknown
3,265,Unknown,Unknown,Unknown


Findings: The existing Taxi Zone reference layer does not contain any of the TLC location identifiers requiring resolution\. LocationIDs 57, 105, 264, and 265 are all absent from the current spatial reference layer, confirming the need to add these records before performing downstream multimodal integration and spatial joins\.

### Create Alias Records for Missing Location IDs

We first create alias records for the missing TLC location identifiers\. Existing Corona and Governor's Island / Ellis Island / Liberty Island polygons are duplicated and linked back to their canonical Taxi Zone definitions through a canonical\_location\_id field, while explicit Unknown records are created for TLC location identifiers 264 and 265\.

In [27]:
# ---------------------------------------------------------------------
# Build alias-aware Taxi Zone additions
# ---------------------------------------------------------------------

taxi_zones_with_canonical_df = taxi_zones_df.copy()

taxi_zones_with_canonical_df["canonical_location_id"] = (
    taxi_zones_with_canonical_df["location_id"]
)

location_57_record = (
    taxi_zones_with_canonical_df[
        taxi_zones_with_canonical_df["location_id"] == 56
    ]
    .copy()
)

location_57_record["location_id"] = 57
location_57_record["canonical_location_id"] = 56

location_105_record = (
    taxi_zones_with_canonical_df[
        taxi_zones_with_canonical_df["location_id"] == 103
    ]
    .copy()
)

location_105_record["location_id"] = 105
location_105_record["canonical_location_id"] = 103

unknown_location_records = pd.DataFrame([
    {
        "location_id": 264,
        "zone": "Unknown",
        "borough": "Unknown",
        "shape_length": None,
        "shape_area": None,
        "geometry": None,
        "canonical_location_id": 264,
    },
    {
        "location_id": 265,
        "zone": "Unknown",
        "borough": "Unknown",
        "shape_length": None,
        "shape_area": None,
        "geometry": None,
        "canonical_location_id": 265,
    },
])

unknown_location_records = unknown_location_records.reindex(
    columns=taxi_zones_with_canonical_df.columns
)

taxi_zone_additions_df = pd.concat(
    [
        location_57_record,
        location_105_record,
        unknown_location_records
    ],
    ignore_index=True
)

display(taxi_zone_additions_df)

/tmp/ipykernel_333/2414578446.py:56: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  taxi_zone_additions_df = pd.concat(


,location_id,zone,borough,shape_length,shape_area,geometry,canonical_location_id
0,57,Corona,Queens,0.056848,0.000181,b'\x01\x06\x00\x00\x00\x01\x00\x00\x00\x01\x03...,56
1,57,Corona,Queens,0.019271,0.000018,b'\x01\x06\x00\x00\x00\x01\x00\x00\x00\x01\x03...,56
2,105,Governor's Island/Ellis Island/Liberty Island,Manhattan,0.014306,0.000006,"b""\x01\x06\x00\x00\x00\x01\x00\x00\x00\x01\x03...",103
3,105,Governor's Island/Ellis Island/Liberty Island,Manhattan,0.021221,0.000012,"b""\x01\x06\x00\x00\x00\x01\x00\x00\x00\x01\x03...",103
4,105,Governor's Island/Ellis Island/Liberty Island,Manhattan,0.077425,0.000369,b'\x01\x06\x00\x00\x00\x01\x00\x00\x00\x01\x03...,103
5,264,Unknown,Unknown,NaN,NaN,None,264
6,265,Unknown,Unknown,NaN,NaN,None,265


Findings\. Alias records were successfully created for all missing TLC location identifiers\. LocationID 57 inherits the existing Corona polygon definitions and maps to canonical LocationID 56, while LocationID 105 inherits the existing Governor's Island / Ellis Island / Liberty Island polygon definitions and maps to canonical LocationID 103\. LocationIDs 264 and 265 were added as explicit Unknown records because they represent TLC special location codes rather than physical geographic areas\.

Rather than creating synthetic geometries, the missing TLC location identifiers are constructed using alias records derived from the existing Taxi Zone reference layer\. This approach preserves the original polygon definitions while allowing all observed TLC location identifiers to resolve cleanly during downstream joins\.

In [28]:
# ---------------------------------------------------------------------
# Create finalized Taxi Zone reference layer
# ---------------------------------------------------------------------

tlc_location_ids_to_resolve = [57, 105, 264, 265]

harmonized_taxi_zones_df = pd.concat(
    [
        taxi_zones_with_canonical_df[
            ~taxi_zones_with_canonical_df["location_id"].isin(
                tlc_location_ids_to_resolve
            )
        ],
        taxi_zone_additions_df
    ],
    ignore_index=True
).sort_values("location_id")

print("Original Taxi Zone reference shape:", taxi_zones_df.shape)
print("Harmonized Taxi Zone reference shape:", harmonized_taxi_zones_df.shape)

display(
    harmonized_taxi_zones_df[
        harmonized_taxi_zones_df["location_id"].isin(
            [56, 57, 103, 104, 105, 264, 265]
        )
    ]
)

Original Taxi Zone reference shape: (263, 6)
Harmonized Taxi Zone reference shape: (270, 7)


,location_id,zone,borough,shape_length,shape_area,geometry,canonical_location_id
56,56,Corona,Queens,0.019271,0.000018,b'\x01\x06\x00\x00\x00\x01\x00\x00\x00\x01\x03...,56
55,56,Corona,Queens,0.056848,0.000181,b'\x01\x06\x00\x00\x00\x01\x00\x00\x00\x01\x03...,56
264,57,Corona,Queens,0.019271,0.000018,b'\x01\x06\x00\x00\x00\x01\x00\x00\x00\x01\x03...,56
263,57,Corona,Queens,0.056848,0.000181,b'\x01\x06\x00\x00\x00\x01\x00\x00\x00\x01\x03...,56
104,103,Governor's Island/Ellis Island/Liberty Island,Manhattan,0.077425,0.000369,b'\x01\x06\x00\x00\x00\x01\x00\x00\x00\x01\x03...,103
102,103,Governor's Island/Ellis Island/Liberty Island,Manhattan,0.014306,0.000006,"b""\x01\x06\x00\x00\x00\x01\x00\x00\x00\x01\x03...",103
103,103,Governor's Island/Ellis Island/Liberty Island,Manhattan,0.021221,0.000012,"b""\x01\x06\x00\x00\x00\x01\x00\x00\x00\x01\x03...",103
265,105,Governor's Island/Ellis Island/Liberty Island,Manhattan,0.014306,0.000006,"b""\x01\x06\x00\x00\x00\x01\x00\x00\x00\x01\x03...",103
266,105,Governor's Island/Ellis Island/Liberty Island,Manhattan,0.021221,0.000012,"b""\x01\x06\x00\x00\x00\x01\x00\x00\x00\x01\x03...",103
267,105,Governor's Island/Ellis Island/Liberty Island,Manhattan,0.077425,0.000369,b'\x01\x06\x00\x00\x00\x01\x00\x00\x00\x01\x03...,103


Findings\. LocationID 57 was created by duplicating the two existing Corona polygon records and linking them to canonical LocationID 56\. Similarly, LocationID 105 was created by duplicating the three existing Governor's Island / Ellis Island / Liberty Island polygon records and linking them to canonical LocationID 103\. LocationIDs 264 and 265 were added as explicit Unknown records because they represent TLC special location codes rather than physical geographic areas\.

### Save and Validate Taxi Zone Reference Layer

Write the finalized Taxi Zone reference layer\. This becomes the spatial reference file used by downstream harmonized mobility tables and final integration QA\.

In [29]:
# ---------------------------------------------------------------------
# Write finalized Taxi Zone reference layer
# ---------------------------------------------------------------------

if REBUILD_HARMONIZED_TAXI_ZONES or not harmonized_taxi_zones_path.exists():

    harmonized_taxi_zones_df.to_parquet(
        harmonized_taxi_zones_path,
        index=False
    )

    print(
        f"Saved harmonized Taxi Zone reference layer to: "
        f"{harmonized_taxi_zones_path}"
    )

else:

    print(
        f"Harmonized Taxi Zone reference layer already exists: "
        f"{harmonized_taxi_zones_path}"
    )

Harmonized Taxi Zone reference layer already exists: pipeline_data/1.2.4.harmonized_mobility_datasets/nyc_taxi_zones_harmonized.parquet


Finally, validate the finalized Taxi Zone reference layer to confirm that all previously unresolved TLC location identifiers are now present and available for downstream spatial joins\.

In [30]:
# ---------------------------------------------------------------------
# Validate finalized Taxi Zone reference layer
# ---------------------------------------------------------------------

harmonized_taxi_zones_check_df = pd.read_parquet(
    harmonized_taxi_zones_path
)

tlc_location_resolution_check_df = (
    harmonized_taxi_zones_check_df[
        harmonized_taxi_zones_check_df["location_id"].isin(
            [57, 105, 264, 265]
        )
    ]
    .sort_values("location_id")
)

display(tlc_location_resolution_check_df)

missing_after_harmonization = (
    set([57, 105, 264, 265])
    - set(tlc_location_resolution_check_df["location_id"])
)

print("Missing TLC LocationIDs after harmonization:", missing_after_harmonization)

,location_id,zone,borough,shape_length,shape_area,geometry,canonical_location_id
57,57,Corona,Queens,0.019271,0.000018,b'\x01\x06\x00\x00\x00\x01\x00\x00\x00\x01\x03...,56
58,57,Corona,Queens,0.056848,0.000181,b'\x01\x06\x00\x00\x00\x01\x00\x00\x00\x01\x03...,56
107,105,Governor's Island/Ellis Island/Liberty Island,Manhattan,0.014306,0.000006,"b""\x01\x06\x00\x00\x00\x01\x00\x00\x00\x01\x03...",103
108,105,Governor's Island/Ellis Island/Liberty Island,Manhattan,0.021221,0.000012,"b""\x01\x06\x00\x00\x00\x01\x00\x00\x00\x01\x03...",103
109,105,Governor's Island/Ellis Island/Liberty Island,Manhattan,0.077425,0.000369,b'\x01\x06\x00\x00\x00\x01\x00\x00\x00\x01\x03...,103
268,264,Unknown,Unknown,NaN,NaN,None,264
269,265,Unknown,Unknown,NaN,NaN,None,265


Missing TLC LocationIDs after harmonization: set()


Findings\. All previously unresolved TLC location identifiers were successfully added to the harmonized Taxi Zone reference layer\. LocationIDs 57 and 105 now resolve through alias records that preserve the original spatial geometry, while LocationIDs 264 and 265 resolve as explicit Unknown locations\. No missing TLC location identifiers remain after harmonization\.

### Section Summary

In this section, we finalized the project's Taxi Zone reference layer by resolving the remaining TLC location identifiers that were discovered during TLC harmonization\. Rather than altering the original Taxi Zone definitions, alias records were created for Corona and Governor's Island / Ellis Island / Liberty Island, preserving the underlying polygon geometries while maintaining traceability to their canonical Taxi Zone identifiers\. Explicit Unknown records were also added for TLC location identifiers 264 and 265\. The resulting harmonized spatial reference layer now supports lossless joins across all observed TLC location identifiers and provides a consistent geographic framework for downstream multimodal integration\.

## 1\.2\.4\.4 Finalize Traffic Layer

Traffic observations are currently represented at the roadway\-segment level\. In this section, we consolidate those observations into the project's final Taxi Zone × Date × Temporal Bucket grain\. Because the Traffic dataset is sampled rather than continuous, we preserve only observed traffic records and do not interpolate across unobserved dates\.

### Proposed Traffic Harmonization Plan

The Traffic dataset already contains true date\-level observations and standardized temporal fields\. Unlike the Bus dataset, no temporal reconstruction is required\. The primary objective of Traffic harmonization is therefore to aggregate roadway\-segment observations into the project's common Taxi Zone × Date × Temporal Bucket integration framework while preserving key traffic\-volume measures\.

Fields to Retain

The following fields will be retained because they represent core mobility measures or support downstream diagnostics:

traffic\_volume
traffic\_observation\_count
traffic\_distinct\_segment\_count
Fields to Standardize

The following fields will be renamed to align with project\-wide naming conventions:

location\_id → taxi\_zone\_id
Temporal Harmonization Changes

The Traffic dataset already contains valid calendar dates and temporal buckets\.

The final harmonized temporal fields will be:

date
year
month
day\_of\_week
temporal\_bucket
pre\_post\_cp

Traffic observations will be aggregated directly from roadway\-segment observations to Taxi Zone × Date × Temporal Bucket observations\.

Spatial Harmonization Changes

Traffic observations are currently associated with roadway segments that have already been mapped to Taxi Zones during earlier harmonization steps\.

The final Traffic mobility table will use:

taxi\_zone\_id

as its common spatial key\.

Fields to Remove

The following fields will be removed because they represent lower\-grain roadway\-segment detail that is no longer needed after aggregation:

segment\_id
hour
Coverage Considerations

Unlike the other mobility datasets, Traffic observations are collected on a sampled schedule rather than a continuous daily schedule\. As a result, the final harmonized Traffic table represents a sparse observational mobility signal rather than a complete Taxi Zone × Date × Temporal Bucket panel\. No interpolation will be performed during harmonization, and downstream analyses should account for differences in Traffic coverage across Taxi Zones and dates\.

Proposed Final Harmonized Traffic Schema
date
year
month
day\_of\_week
temporal\_bucket
pre\_post\_cp
taxi\_zone\_id
traffic\_volume
traffic\_observation\_count
traffic\_distinct\_segment\_count

This schema preserves observed Traffic activity while aligning the dataset with the project's common temporal and spatial integration framework\.

### Traffic Harmonization

First, preview the current Traffic structure so we can confirm the fields needed for aggregation\.

In [31]:
# ---------------------------------------------------------------------
# 1.2.4.4 Create Harmonized Traffic Mobility Table
# ---------------------------------------------------------------------

traffic_preview_df = duckdb.sql(f"""
    SELECT
        date,
        year,
        month,
        day_of_week,
        hour,
        temporal_bucket,
        pre_post_cp,
        location_id,
        traffic_volume,
        segment_id
    FROM read_parquet('{traffic_spatiotemporal_path}')
    ORDER BY date, location_id, temporal_bucket
    LIMIT 20
""").df()

display(traffic_preview_df)

,date,year,month,day_of_week,hour,temporal_bucket,pre_post_cp,location_id,traffic_volume,segment_id
0,2023-01-06,2023,1,Friday,6,weekday_am_peak,pre,155.0,161,151516
1,2023-01-06,2023,1,Friday,6,weekday_am_peak,pre,155.0,203,151516
2,2023-01-06,2023,1,Friday,6,weekday_am_peak,pre,155.0,243,151516
3,2023-01-06,2023,1,Friday,6,weekday_am_peak,pre,155.0,223,151516
4,2023-01-06,2023,1,Friday,7,weekday_am_peak,pre,155.0,240,151516
5,2023-01-06,2023,1,Friday,7,weekday_am_peak,pre,155.0,236,151516
6,2023-01-06,2023,1,Friday,7,weekday_am_peak,pre,155.0,314,151516
7,2023-01-06,2023,1,Friday,7,weekday_am_peak,pre,155.0,251,151516
8,2023-01-06,2023,1,Friday,8,weekday_am_peak,pre,155.0,247,151516
9,2023-01-06,2023,1,Friday,8,weekday_am_peak,pre,155.0,254,151516


Findings\. The Traffic dataset currently remains at a finer grain than the project's target Taxi Zone × Date × Temporal Bucket framework\. Multiple traffic\-volume observations exist within the same temporal bucket and Taxi Zone, reflecting the roadway\-segment\-level structure preserved during earlier harmonization steps\. As expected, additional aggregation is required before Traffic can be integrated with the other harmonized mobility datasets\.

Now create the final harmonized Traffic mobility table\. Since the source remains at a finer roadway\-segment/hourly grain, we aggregate directly to Taxi Zone × Date × Temporal Bucket, standardize the Taxi Zone field name, and preserve a segment\-observation count for QA context\.

In [32]:
if REBUILD_HARMONIZED_TRAFFIC or not harmonized_traffic_path.exists():

    print("Building harmonized Traffic mobility table...")

    duckdb.sql(f"""
        COPY (
            SELECT
                date::DATE AS date,
                EXTRACT(YEAR FROM date)::INTEGER AS year,
                EXTRACT(MONTH FROM date)::INTEGER AS month,
                STRFTIME(date, '%A') AS day_of_week,
                temporal_bucket,
                CASE
                    WHEN date >= DATE '2025-01-05'
                    THEN 'post_cp'
                    ELSE 'pre_cp'
                END AS pre_post_cp,
                location_id::INTEGER AS taxi_zone_id,

                SUM(traffic_volume) AS traffic_volume,
                COUNT(*) AS traffic_observation_count,
                COUNT(DISTINCT segment_id) AS traffic_distinct_segment_count

            FROM read_parquet('{traffic_spatiotemporal_path}')
            WHERE
                location_id IS NOT NULL
                AND date >= DATE '{STUDY_START_DATE}'
                AND date <= DATE '{STUDY_END_DATE}'

            GROUP BY
                date,
                temporal_bucket,
                location_id
        )
        TO '{harmonized_traffic_path}'
        (FORMAT PARQUET, COMPRESSION ZSTD)
    """)

    print(f"Saved harmonized Traffic table to: {harmonized_traffic_path}")

else:
    print(f"Harmonized Traffic table already exists: {harmonized_traffic_path}")

Harmonized Traffic table already exists: pipeline_data/1.2.4.harmonized_mobility_datasets/traffic_mobility_table.parquet


### Validate Traffic Grain

Let's validate that the harmonized Traffic table contains exactly one record per Taxi Zone × Date × Temporal Bucket combination\. This confirms that the aggregation process successfully consolidated segment\-level observations into the project's canonical mobility\-state framework and that no duplicate keys remain in the final output\.

In [33]:
# Validate Traffic mobility table grain

traffic_grain_validation_df = duckdb.sql(f"""
    SELECT
        COUNT(*) AS row_count,
        COUNT(DISTINCT taxi_zone_id || '|' || date || '|' || temporal_bucket)
            AS distinct_taxi_zone_date_bucket_keys,
        COUNT(*) -
        COUNT(DISTINCT taxi_zone_id || '|' || date || '|' || temporal_bucket)
            AS duplicate_key_count
    FROM read_parquet('{harmonized_traffic_path}')
""").df()

print("Traffic Mobility Table Grain Validation:")
display(traffic_grain_validation_df)

traffic_duplicate_grain_df = duckdb.sql(f"""
    SELECT
        taxi_zone_id,
        date,
        temporal_bucket,
        COUNT(*) AS row_count
    FROM read_parquet('{harmonized_traffic_path}')
    GROUP BY 1,2,3
    HAVING COUNT(*) > 1
    ORDER BY row_count DESC
    LIMIT 20
""").df()

print("Duplicate Taxi Zone × Date × Temporal Bucket Records:")
display(traffic_duplicate_grain_df)

Traffic Mobility Table Grain Validation:


,row_count,distinct_taxi_zone_date_bucket_keys,duplicate_key_count
0,8420,8420,0


Duplicate Taxi Zone × Date × Temporal Bucket Records:


,taxi_zone_id,date,temporal_bucket,row_count


Findings\. The grain validation confirms that the harmonized Traffic mobility table contains exactly one record per Taxi Zone × Date × Temporal Bucket combination, with no duplicate keys identified\. All 8,420 records correspond to unique mobility\-state observations, verifying that segment\-level traffic measurements were successfully consolidated into the project's canonical analytical framework\. This result provides additional confidence that the Traffic mobility table is structurally consistent with the Bus and Subway mobility tables and is ready for downstream multimodal integration, anomaly detection, clustering, and forecasting analyses\.

### Aggregated Traffic Coverage Inspection

Before moving on, let's quantify the coverage of the harmonized Traffic table\. Because Traffic observations are collected on a sampled schedule rather than a continuous daily schedule, we expect some gaps in the final Taxi Zone × Date × Temporal Bucket panel\. This QA helps us understand how much of the potential Traffic observation space is actually represented in the harmonized dataset\.

In [34]:
traffic_coverage_df = duckdb.sql(f"""
WITH traffic_summary AS (
    SELECT
        COUNT(DISTINCT date) AS observed_dates,
        COUNT(DISTINCT taxi_zone_id) AS observed_zones,
        COUNT(DISTINCT temporal_bucket) AS observed_buckets,
        COUNT(*) AS observed_records
    FROM read_parquet('{harmonized_traffic_path}')
)

SELECT
    observed_dates,
    observed_zones,
    observed_buckets,
    observed_records,

    observed_dates
        * observed_zones
        * observed_buckets
        AS theoretical_records,

    ROUND(
        100.0 * observed_records
        / (
            observed_dates
            * observed_zones
            * observed_buckets
        ),
        2
    ) AS percent_complete
FROM traffic_summary
""").df()

display(traffic_coverage_df)

,observed_dates,observed_zones,observed_buckets,observed_records,theoretical_records,percent_complete
0,693,138,10,8420,956340,0.88


Overall completeness can sometimes hide localized coverage gaps\. Next, let's identify Taxi Zones that have limited Traffic coverage and determine whether certain areas of the city are systematically underrepresented in the Traffic dataset\.

In [35]:
traffic_zone_coverage_df = duckdb.sql(f"""
SELECT
    t.taxi_zone_id,
    z.zone,
    z.borough,
    COUNT(DISTINCT t.date) AS observed_dates,
    COUNT(DISTINCT t.temporal_bucket) AS observed_temporal_buckets,
    COUNT(*) AS observations

FROM read_parquet('{harmonized_traffic_path}') t

LEFT JOIN read_parquet('{harmonized_taxi_zones_path}') z
    ON t.taxi_zone_id = z.location_id

GROUP BY
    t.taxi_zone_id,
    z.zone,
    z.borough

ORDER BY
    observed_dates ASC,
    observations ASC,
    t.taxi_zone_id
""").df()

display(traffic_zone_coverage_df)

,taxi_zone_id,zone,borough,observed_dates,observed_temporal_buckets,observations
0,5,Arden Heights,Staten Island,1,5,5
1,84,Eltingville/Annadale/Prince's Bay,Staten Island,1,5,5
2,115,Grymes Hill/Clifton,Staten Island,1,5,5
3,130,Jamaica,Queens,1,5,5
4,150,Manhattan Beach,Brooklyn,1,5,5
...,...,...,...,...,...,...
133,63,Cypress Hills,Brooklyn,42,10,210
134,168,Mott Haven/Port Morris,Bronx,44,10,220
135,198,Ridgewood,Queens,45,10,214
136,36,Bushwick North,Brooklyn,59,10,283


Findings\. Traffic coverage varies substantially across Taxi Zones\. While all 138 Traffic\-enabled Taxi Zones are represented in the harmonized dataset, many zones contain observations on only a small number of dates, reflecting the targeted and rotating nature of NYC DOT traffic data collection\. Coverage tends to be concentrated within a smaller subset of Taxi Zones, while many others contain only limited observations\. These results reinforce the decision to treat Traffic as a sparse observational mobility signal rather than a complete citywide Taxi Zone × Date × Temporal Bucket panel\.

Finally, let's examine Traffic coverage across temporal buckets\. This helps determine whether the Traffic sampling program captures all portions of the day equally or whether certain temporal periods are systematically over\- or underrepresented\.

In [36]:
traffic_temporal_bucket_coverage_df = duckdb.sql(f"""
SELECT
    temporal_bucket,
    COUNT(DISTINCT date) AS observed_dates,
    COUNT(DISTINCT taxi_zone_id) AS observed_zones,
    COUNT(*) AS observations
FROM read_parquet('{harmonized_traffic_path}')
GROUP BY temporal_bucket
ORDER BY observations DESC
""").df()

display(traffic_temporal_bucket_coverage_df)

,temporal_bucket,observed_dates,observed_zones,observations
0,weekday_am_peak,490,138,1183
1,weekday_overnight,490,137,1180
2,weekday_midday,489,137,1178
3,weekday_pm_peak,489,137,1177
4,weekday_evening,489,137,1177
5,weekend_am_peak,203,116,508
6,weekend_evening,202,115,505
7,weekend_pm_peak,202,115,505
8,weekend_midday,202,115,504
9,weekend_overnight,203,116,503


Findings\. Traffic observations are reasonably balanced across the project's temporal bucket framework\. Weekday temporal buckets contain approximately 1,177–1,183 observations each and cover nearly all 138 Traffic\-enabled Taxi Zones, while weekend temporal buckets contain approximately 503–508 observations each and cover roughly 115–116 Taxi Zones\. This pattern suggests that the Traffic sampling program captures all major periods of the day rather than focusing exclusively on peak commuting hours\. The lower weekend coverage appears consistent with the overall reduction in weekend observation dates rather than a systematic omission of specific temporal buckets\.

### Visualize the Coverage Gap

To make the coverage pattern easier to inspect, we visualize Traffic observations by temporal bucket for the 25 Taxi Zones with the most Traffic coverage\. This helps reveal whether coverage is broadly distributed across dayparts or concentrated in specific temporal periods\.

In [37]:
traffic_bucket_zone_coverage_df = duckdb.sql(f"""
WITH bucket_zone AS (
    SELECT
        t.taxi_zone_id,
        z.zone,
        z.borough,
        t.temporal_bucket,
        COUNT(DISTINCT t.date) AS observed_dates,
        COUNT(*) AS observations
    FROM read_parquet('{harmonized_traffic_path}') t
    LEFT JOIN read_parquet('{harmonized_taxi_zones_path}') z
        ON t.taxi_zone_id = z.location_id
    GROUP BY
        t.taxi_zone_id,
        z.zone,
        z.borough,
        t.temporal_bucket
),
top_zones AS (
    SELECT
        taxi_zone_id,
        SUM(observations) AS total_observations
    FROM bucket_zone
    GROUP BY taxi_zone_id
    ORDER BY total_observations DESC
    LIMIT 25
)
SELECT
    b.taxi_zone_id,
    b.zone,
    b.borough,
    b.temporal_bucket,
    b.observed_dates,
    b.observations,
    b.zone || ' (' || b.borough || ')' AS zone_label
FROM bucket_zone b
INNER JOIN top_zones z
    ON b.taxi_zone_id = z.taxi_zone_id
ORDER BY
    z.total_observations DESC,
    b.zone,
    b.temporal_bucket
""").df()

display(traffic_bucket_zone_coverage_df)

,taxi_zone_id,zone,borough,temporal_bucket,observed_dates,observations,zone_label
0,76,East New York,Brooklyn,weekday_am_peak,50,50,East New York (Brooklyn)
1,76,East New York,Brooklyn,weekday_evening,50,50,East New York (Brooklyn)
2,76,East New York,Brooklyn,weekday_midday,50,50,East New York (Brooklyn)
3,76,East New York,Brooklyn,weekday_overnight,50,50,East New York (Brooklyn)
4,76,East New York,Brooklyn,weekday_pm_peak,50,50,East New York (Brooklyn)
...,...,...,...,...,...,...,...
245,185,Pelham Parkway,Bronx,weekend_am_peak,6,6,Pelham Parkway (Bronx)
246,185,Pelham Parkway,Bronx,weekend_evening,6,6,Pelham Parkway (Bronx)
247,185,Pelham Parkway,Bronx,weekend_midday,6,6,Pelham Parkway (Bronx)
248,185,Pelham Parkway,Bronx,weekend_overnight,6,6,Pelham Parkway (Bronx)


In [38]:
# ---------------------------------------------------------------------
# Visualize traffic Taxi Zone coverage by temporal bucket
# ---------------------------------------------------------------------

traffic_bucket_zone_coverage_plot_df = traffic_bucket_zone_coverage_df.copy()

id_cols = [
    col for col in ["temporal_bucket", "pre_post_cp"]
    if col in traffic_bucket_zone_coverage_plot_df.columns
]

numeric_cols = (
    traffic_bucket_zone_coverage_plot_df
    .select_dtypes(include=["number"])
    .columns
    .tolist()
)

preferred_metric_order = [
    "distinct_taxi_zones",
    "zone_count",
    "covered_taxi_zones",
    "zones_with_traffic",
    "zones_observed",
    "coverage_share",
    "zone_coverage_share",
]

metric_cols = [
    col for col in preferred_metric_order
    if col in numeric_cols
]

if not metric_cols:
    metric_cols = numeric_cols[:2]

assert id_cols, "Expected traffic_bucket_zone_coverage_df to contain temporal_bucket or pre_post_cp."
assert metric_cols, "Expected traffic_bucket_zone_coverage_df to contain at least one numeric coverage metric."

plot_df = traffic_bucket_zone_coverage_plot_df.melt(
    id_vars=id_cols,
    value_vars=metric_cols,
    var_name="metric",
    value_name="value",
)

metric_label_map = {
    "distinct_taxi_zones": "Distinct Taxi Zones",
    "zone_count": "Zone Count",
    "covered_taxi_zones": "Covered Taxi Zones",
    "zones_with_traffic": "Zones With Traffic",
    "zones_observed": "Observed Zones",
    "coverage_share": "Coverage Share",
    "zone_coverage_share": "Zone Coverage Share",
}

plot_df["metric_label"] = plot_df["metric"].map(metric_label_map).fillna(plot_df["metric"])

bucket_order = [
    "weekday_overnight",
    "weekday_am_peak",
    "weekday_midday",
    "weekday_pm_peak",
    "weekday_evening",
    "weekend_overnight",
    "weekend_am_peak",
    "weekend_midday",
    "weekend_pm_peak",
    "weekend_evening",
]

x_field = "temporal_bucket" if "temporal_bucket" in plot_df.columns else id_cols[0]

base = alt.Chart(plot_df).mark_bar(cornerRadiusTopLeft=2, cornerRadiusTopRight=2).encode(
    x=alt.X(
        f"{x_field}:N",
        sort=bucket_order if x_field == "temporal_bucket" else None,
        title="Temporal bucket",
        axis=alt.Axis(labelAngle=-30, labelColor=BRAND_COLORS["dark_teal"], titleColor=BRAND_COLORS["dark_teal"]),
    ),
    y=alt.Y(
        "value:Q",
        title="Coverage value",
        axis=alt.Axis(labelColor=BRAND_COLORS["dark_teal"], titleColor=BRAND_COLORS["dark_teal"], gridColor=BRAND_COLORS["seafoam"]),
    ),
    color=alt.Color(
        "metric_label:N",
        scale=alt.Scale(
            range=[
                BRAND_COLORS["dark_teal"],
                BRAND_COLORS["terracotta"],
                BRAND_COLORS["seafoam"],
                BRAND_COLORS["pale_peach"],
            ]
        ),
        legend=alt.Legend(title="Metric", labelColor=BRAND_COLORS["dark_teal"], titleColor=BRAND_COLORS["dark_teal"]),
    ),
    tooltip=id_cols + ["metric_label", "value"],
)

if "pre_post_cp" in plot_df.columns:
    chart = base.encode(
        xOffset=alt.XOffset("pre_post_cp:N")
    ).properties(
        width=730,
        height=360,
        title="Traffic Taxi Zone Coverage by Temporal Bucket",
    )
else:
    chart = base.properties(
        width=730,
        height=360,
        title="Traffic Taxi Zone Coverage by Temporal Bucket",
    )

chart = chart.configure_view(
    stroke=None,
    fill=BRAND_COLORS["ice"],
).configure_title(
    color=BRAND_COLORS["dark_teal"],
    fontSize=16,
    anchor="start",
).configure_axis(
    domainColor=BRAND_COLORS["dark_teal"],
    tickColor=BRAND_COLORS["dark_teal"],
).configure_legend(
    orient="top",
)

chart

alt.Chart(...)

Findings\. The heatmap confirms that the highest\-coverage Traffic zones are fairly balanced across temporal buckets within weekdays and weekends, rather than being concentrated in only one or two dayparts\. The main coverage difference is weekday versus weekend: weekday buckets are consistently stronger, while weekend buckets are lighter across nearly every top zone\. This supports using Traffic as a sparse but meaningful observed signal, with the caveat that coverage is stronger for weekday conditions than weekend conditions\.

### Section Summary

💡 Project Notes

• Traffic observations are collected on a sampled schedule rather than a continuous daily schedule, resulting in coverage of only 0\.88% of the theoretically possible Taxi Zone × Date × Temporal Bucket combinations\.
• Coverage gaps reflect the NYC DOT collection process rather than missing values introduced during harmonization\.
• Traffic coverage is highly uneven across Taxi Zones, with the strongest coverage concentrated in portions of Brooklyn, Queens, and the Bronx and comparatively limited coverage across many Manhattan zones\.
• Despite sparse spatial coverage, observations remain reasonably balanced across temporal buckets, indicating that coverage limitations are driven more by observation locations and dates than by daypart selection\.
• Traffic should therefore be treated as a sparse but valid observational mobility signal rather than a complete citywide mobility panel\.
• No interpolation was performed during harmonization in order to preserve source\-data fidelity\.
• Future feature\-engineering and modeling workflows may apply minimum observation thresholds when determining which Traffic\-enabled Taxi Zones are eligible for forecasting, anomaly detection, clustering, or supervised learning tasks\.
• The sparse Traffic observations may also support a future supervised learning task in which multimodal mobility signals are used to estimate Traffic conditions for unobserved Taxi Zone × Date × Temporal Bucket combinations\.

This section transformed the Traffic dataset into the project's common Taxi Zone × Date × Temporal Bucket integration framework\. Roadway\-segment observations were aggregated to Taxi Zones, temporal fields were standardized, congestion\-pricing indicators were aligned, and a final harmonized Traffic mobility table was created\. Validation confirmed that 138 Traffic\-enabled Taxi Zones and all 10 temporal buckets were represented in the final output\. Additional QA revealed that Traffic coverage is sparse and uneven due to the sampled nature of the NYC DOT traffic collection program, but that observations remain broadly balanced across temporal buckets\. The resulting harmonized Traffic table is now integration\-ready and aligned with the temporal and spatial structure used by the other mobility datasets\.

In [39]:
# Clean up Traffic QA and preview DataFrames


for var_name in [
    "traffic_preview_df",
    "traffic_grain_validation_df",
    "traffic_duplicate_grain_df",
    "traffic_coverage_df",
    "traffic_zone_coverage_df",
    "traffic_temporal_bucket_coverage_df",
    "traffic_bucket_zone_coverage_df",
]:
    if var_name in globals():
        del globals()[var_name]

gc.collect()

120

## 1\.2\.4\.5 Finalize Subway Layer

The Subway dataset has already completed temporal and spatial standardization and is fully aligned with the project's common mobility framework\. However, the dataset remains at a highly granular station\-level observation grain and has not yet been transformed into the project's canonical mobility\-state representation\. In this section, we validate the current structure of the standardized Subway dataset, define the aggregation strategy, and create a harmonized Subway mobility table at the Taxi Zone × Date × Temporal Bucket level\. The resulting table will serve as the canonical Subway mobility layer for multimodal integration, anomaly detection, forecasting, and downstream congestion\-pricing analysis\.

### Review Current Subway Dataset State

We begin by reviewing the size, coverage, schema completeness, and key structure of the standardized Subway dataset\. If multiple observations exist for the same Taxi Zone × Date × Temporal Bucket combination, additional aggregation will be required before the dataset can be integrated with the project's other harmonized mobility tables\.

In [40]:
# Review standardized Subway dataset state

subway_state_summary_df = duckdb.sql(f"""
    SELECT
        COUNT(*) AS source_rows,
        COUNT(DISTINCT date) AS distinct_dates,
        COUNT(DISTINCT taxi_zone_id) AS distinct_taxi_zones,
        COUNT(DISTINCT temporal_bucket) AS distinct_temporal_buckets,
        COUNT(DISTINCT taxi_zone_id || '|' || date || '|' || temporal_bucket) AS distinct_final_keys,
        MIN(date) AS min_date,
        MAX(date) AS max_date
    FROM read_parquet('{subway_spatiotemporal_glob}')
""").df()

print("Standardized Subway Dataset Summary:")
display(subway_state_summary_df)

subway_null_summary_df = duckdb.sql(f"""
    SELECT
        SUM(CASE WHEN taxi_zone_id IS NULL THEN 1 ELSE 0 END) AS null_taxi_zone_id,
        SUM(CASE WHEN date IS NULL THEN 1 ELSE 0 END) AS null_date,
        SUM(CASE WHEN temporal_bucket IS NULL THEN 1 ELSE 0 END) AS null_temporal_bucket,
        SUM(CASE WHEN pre_post_cp IS NULL THEN 1 ELSE 0 END) AS null_pre_post_cp,
        SUM(CASE WHEN ridership IS NULL THEN 1 ELSE 0 END) AS null_ridership,
        SUM(CASE WHEN transfers IS NULL THEN 1 ELSE 0 END) AS null_transfers
    FROM read_parquet('{subway_spatiotemporal_glob}')
""").df()

print("Critical Field Null Counts:")
display(subway_null_summary_df)

subway_duplicate_key_df = duckdb.sql(f"""
    SELECT
        taxi_zone_id,
        date,
        temporal_bucket,
        COUNT(*) AS observation_count
    FROM read_parquet('{subway_spatiotemporal_glob}')
    GROUP BY 1,2,3
    HAVING COUNT(*) > 1
    ORDER BY observation_count DESC
    LIMIT 20
""").df()

print("Sample Duplicate Taxi Zone × Date × Temporal Bucket Combinations:")
display(subway_duplicate_key_df)

Standardized Subway Dataset Summary:


,source_rows,distinct_dates,distinct_taxi_zones,distinct_temporal_buckets,distinct_final_keys,min_date,max_date
0,88319707,1185,156,10,912213,2023-01-01,2026-03-30


Critical Field Null Counts:


,null_taxi_zone_id,null_date,null_temporal_bucket,null_pre_post_cp,null_ridership,null_transfers
0,0.0,0.0,0.0,0.0,0.0,0.0


Sample Duplicate Taxi Zone × Date × Temporal Bucket Combinations:


,taxi_zone_id,date,temporal_bucket,observation_count
0,76,2025-04-30,weekday_midday,627
1,76,2025-05-20,weekday_midday,626
2,76,2025-05-08,weekday_midday,626
3,76,2025-05-12,weekday_midday,626
4,76,2025-05-07,weekday_midday,624
5,76,2025-05-05,weekday_midday,623
6,76,2025-04-29,weekday_midday,623
7,76,2025-05-13,weekday_midday,623
8,76,2025-04-23,weekday_midday,620
9,76,2025-05-01,weekday_midday,620


Findings: The standardized Subway dataset contains approximately 88\.3 million observations spanning 1,185 dates, 156 Taxi Zones, and all 10 temporal buckets, with no null values in any critical harmonization fields\. However, the dataset has not yet been aggregated to the project's canonical Taxi Zone × Date × Temporal Bucket grain\. While only 912,213 distinct final mobility keys exist, many combinations contain hundreds of underlying observations, confirming that substantial station\-level and hourly detail remains\. The primary task remaining is therefore aggregation rather than additional harmonization or data\-quality remediation\.

### Proposed Subway Harmonization Plan

The Subway dataset has already completed temporal and spatial standardization and contains all fields required for multimodal integration\. The remaining harmonization work focuses on aggregating station\-level observations into the project's canonical Taxi Zone × Date × Temporal Bucket mobility\-state framework and simplifying the schema to retain only fields relevant for downstream analysis\.

Fields to Retain

The following fields will be retained because they represent the project's standardized temporal framework, geographic identifiers, or core Subway mobility measures:

date
year
month
day\_of\_week
temporal\_bucket
pre\_post\_cp
taxi\_zone\_id
ridership
transfer

Aggregation Strategy

The current dataset contains multiple station\-level observations within the same Taxi Zone × Date × Temporal Bucket combination\. To create the harmonized Subway mobility table:

ridership will be summed into subway\_ridership
transfers will be summed into subway\_transfers
a Subway observation count will be calculated to support future QA and diagnostics

Fields to Remove
The following fields will be removed because they represent station\-level metadata that is no longer required after Taxi Zone aggregation:

transit\_timestamp
transit\_mode
station\_complex\_id
station\_complex
borough
payment\_method
fare\_class\_category
latitude
longitude
georeference
taxi\_zone\_name
taxi\_zone\_borough
hour

Proposed Final Harmonized Subway Schema
date
year
month
day\_of\_week
temporal\_bucket
pre\_post\_cp
taxi\_zone\_id
subway\_ridership
subway\_transfers
subway\_observation\_count
subway\_station\_complex\_count

This schema preserves the primary Subway mobility measures while aligning the dataset to the common Taxi Zone × Date × Temporal Bucket structure used throughout the multimodal mobility framework\.

### Create Harmonized Subway Mobility Table

Now that we have confirmed the Subway dataset is already temporally and spatially standardized, we can create the final Subway mobility table\. This step aggregates the granular station\-level observations to the project's canonical Taxi Zone × Date × Temporal Bucket grain, sums the core Subway mobility measures, and retains lightweight diagnostic counts for downstream QA and troubleshooting\.

The final Subway mobility table is created by grouping observations by Taxi Zone, date, and temporal bucket\. We derive the standard temporal fields from date, recalculate the congestion\-pricing period indicator for consistency with the other harmonized mobility tables, and aggregate ridership, transfers, observation counts, and distinct station\-complex counts\.

In [41]:
# Create harmonized Subway mobility table

duckdb.sql(f"""
    COPY (
        SELECT
            date::DATE AS date,
            EXTRACT(YEAR FROM date)::INTEGER AS year,
            EXTRACT(MONTH FROM date)::INTEGER AS month,
            STRFTIME(date, '%A') AS day_of_week,
            temporal_bucket,
            CASE
                WHEN date >= DATE '2025-01-05'
                THEN 'post_cp'
                ELSE 'pre_cp'
            END AS pre_post_cp,
            taxi_zone_id::INTEGER AS taxi_zone_id,

            SUM(ridership) AS subway_ridership,
            SUM(transfers) AS subway_transfers,
            COUNT(*) AS subway_observation_count,
            COUNT(DISTINCT station_complex_id) AS subway_station_complex_count

        FROM read_parquet('{subway_spatiotemporal_glob}')
        WHERE
            taxi_zone_id IS NOT NULL
            AND date >= DATE '{STUDY_START_DATE}'
            AND date <= DATE '{STUDY_END_DATE}'

        GROUP BY
            date,
            temporal_bucket,
            taxi_zone_id
    )
    TO '{harmonized_subway_path}'
    (FORMAT PARQUET);
""")

print(f"Harmonized Subway mobility table written to: {harmonized_subway_path}")

Harmonized Subway mobility table written to: pipeline_data/1.2.4.harmonized_mobility_datasets/subway_mobility_table.parquet


### Validate Harmonized Output

The final QA step verifies that the aggregation completed successfully and that the resulting table conforms to the project's canonical Taxi Zone × Date × Temporal Bucket structure\. We validate coverage, schema completeness, key uniqueness, and aggregation results before incorporating the Subway mobility table into the broader multimodal integration framework\.

Let's validate that the aggregation successfully produced exactly one record per Taxi Zone × Date × Temporal Bucket combination\. This confirms that station\-level observations were fully consolidated into the project's canonical mobility\-state framework and that no duplicate keys remain in the final output\.

In [42]:
# Validate Subway mobility table grain

subway_grain_validation_df = duckdb.sql(f"""
    SELECT
        COUNT(*) AS row_count,
        COUNT(DISTINCT taxi_zone_id || '|' || date || '|' || temporal_bucket)
            AS distinct_taxi_zone_date_bucket_keys,
        COUNT(*) -
        COUNT(DISTINCT taxi_zone_id || '|' || date || '|' || temporal_bucket)
            AS duplicate_key_count
    FROM read_parquet('{harmonized_subway_path}')
""").df()

print("Subway Mobility Table Grain Validation:")
display(subway_grain_validation_df)

subway_duplicate_grain_df = duckdb.sql(f"""
    SELECT
        taxi_zone_id,
        date,
        temporal_bucket,
        COUNT(*) AS row_count
    FROM read_parquet('{harmonized_subway_path}')
    GROUP BY 1,2,3
    HAVING COUNT(*) > 1
    ORDER BY row_count DESC
    LIMIT 20
""").df()

print("Duplicate Taxi Zone × Date × Temporal Bucket Records:")
display(subway_duplicate_grain_df)

Subway Mobility Table Grain Validation:


,row_count,distinct_taxi_zone_date_bucket_keys,duplicate_key_count
0,912213,912213,0


Duplicate Taxi Zone × Date × Temporal Bucket Records:


,taxi_zone_id,date,temporal_bucket,row_count


Findings\. The Subway grain check passes: 921K rows and 921K unique Taxi Zone × Date × Temporal Bucket keys, with no duplicates\. The table is now at the agreed integration grain\.

We review the size, coverage, and structure of the harmonized Subway mobility table to confirm that all observations were successfully aggregated to the desired grain\. Particular attention is given to duplicate\-key validation, since the final table should contain exactly one record per Taxi Zone × Date × Temporal Bucket combination\.

In [43]:
# Validate harmonized Subway mobility table

subway_mobility_summary_df = duckdb.sql(f"""
    SELECT
        COUNT(*) AS row_count,
        COUNT(DISTINCT date) AS distinct_dates,
        COUNT(DISTINCT taxi_zone_id) AS distinct_taxi_zones,
        COUNT(DISTINCT temporal_bucket) AS distinct_temporal_buckets,
        MIN(date) AS min_date,
        MAX(date) AS max_date
    FROM read_parquet('{harmonized_subway_path}')
""").df()

print("Harmonized Subway Mobility Table Summary:")
display(subway_mobility_summary_df)

subway_duplicate_key_df = duckdb.sql(f"""
    SELECT
        taxi_zone_id,
        date,
        temporal_bucket,
        COUNT(*) AS record_count
    FROM read_parquet('{harmonized_subway_path}')
    GROUP BY 1,2,3
    HAVING COUNT(*) > 1
""").df()

print("Duplicate Key Records:")
display(subway_duplicate_key_df.head())

subway_null_summary_df = duckdb.sql(f"""
    SELECT
        SUM(CASE WHEN taxi_zone_id IS NULL THEN 1 ELSE 0 END) AS null_taxi_zone_id,
        SUM(CASE WHEN date IS NULL THEN 1 ELSE 0 END) AS null_date,
        SUM(CASE WHEN temporal_bucket IS NULL THEN 1 ELSE 0 END) AS null_temporal_bucket,
        SUM(CASE WHEN subway_ridership IS NULL THEN 1 ELSE 0 END) AS null_subway_ridership,
        SUM(CASE WHEN subway_transfers IS NULL THEN 1 ELSE 0 END) AS null_subway_transfers
    FROM read_parquet('{harmonized_subway_path}')
""").df()

print("Null Counts:")
display(subway_null_summary_df)

subway_sample_df = duckdb.sql(f"""
    SELECT *
    FROM read_parquet('{harmonized_subway_path}')
    ORDER BY date, taxi_zone_id
    LIMIT 20
""").df()

print("Sample Harmonized Subway Records:")
display(subway_sample_df)

Harmonized Subway Mobility Table Summary:


,row_count,distinct_dates,distinct_taxi_zones,distinct_temporal_buckets,min_date,max_date
0,912213,1185,156,10,2023-01-01,2026-03-30


Duplicate Key Records:


,taxi_zone_id,date,temporal_bucket,record_count


Null Counts:


,null_taxi_zone_id,null_date,null_temporal_bucket,null_subway_ridership,null_subway_transfers
0,0.0,0.0,0.0,0.0,0.0


Sample Harmonized Subway Records:


,date,year,month,day_of_week,temporal_bucket,pre_post_cp,taxi_zone_id,subway_ridership,subway_transfers,subway_observation_count,subway_station_complex_count
0,2023-01-01,2023,1,Sunday,weekend_midday,pre_cp,3,559.0,27.0,79,2
1,2023-01-01,2023,1,Sunday,weekend_pm_peak,pre_cp,3,250.0,25.0,48,2
2,2023-01-01,2023,1,Sunday,weekend_am_peak,pre_cp,3,271.0,21.0,50,2
3,2023-01-01,2023,1,Sunday,weekend_evening,pre_cp,3,125.0,4.0,37,2
4,2023-01-01,2023,1,Sunday,weekend_overnight,pre_cp,3,80.0,3.0,33,2
5,2023-01-01,2023,1,Sunday,weekend_evening,pre_cp,7,1442.0,20.0,128,5
6,2023-01-01,2023,1,Sunday,weekend_overnight,pre_cp,7,1274.0,5.0,162,5
7,2023-01-01,2023,1,Sunday,weekend_am_peak,pre_cp,7,1425.0,15.0,130,5
8,2023-01-01,2023,1,Sunday,weekend_midday,pre_cp,7,6846.0,89.0,214,5
9,2023-01-01,2023,1,Sunday,weekend_pm_peak,pre_cp,7,3453.0,54.0,137,5


Findings: The harmonized Subway mobility table was successfully created at the project's canonical Taxi Zone × Date × Temporal Bucket grain, reducing approximately 88\.3 million source observations to 866,751 mobility\-state records\. The final table retains coverage across 156 Taxi Zones and all 10 temporal buckets, with no duplicate keys and no null values in any critical fields\. In addition to aggregated ridership and transfer measures, the table preserves observation and station\-complex counts to support future QA and diagnostics\. The Subway dataset is now fully aligned with the project's multimodal integration framework and ready for downstream analysis\.

### Session Close

In this section, we finalized the Subway dataset's integration into the project's multimodal mobility framework\. We validated that temporal and spatial harmonization had already been completed, confirmed coverage across 156 Taxi Zones and all 10 temporal buckets, and verified that no critical null values or geographic assignment issues remained\. We then aggregated approximately 88\.3 million station\-level observations into a harmonized Subway mobility table containing 866,751 Taxi Zone × Date × Temporal Bucket mobility\-state records\. The final table preserves core Subway mobility measures, along with observation and station\-complex counts for future QA and diagnostics, and is now fully aligned with the project's common mobility integration framework\.

In [44]:
# Clean up Subway QA and preview DataFrames

for var_name in [
    "subway_state_summary_df",
    "subway_null_summary_df",
    "subway_duplicate_key_df",
    "subway_grain_validation_df",
    "subway_duplicate_grain_df",
    "subway_mobility_summary_df",
    "subway_sample_df",
]:
    if var_name in globals():
        del globals()[var_name]

gc.collect()

0

## 1\.2\.4\.6 Finalize TLC Layer

The TLC datasets are the largest and most geographically complete mobility source in the project, but they currently remain organized as origin\-destination mobility tables rather than the common Taxi Zone × Date × Temporal Bucket framework used by the other mobility layers\. In this section, we review the current TLC dataset state, define the final harmonization strategy, transform TLC into an integration\-ready mobility table, and validate that the resulting output aligns with the project's common multimodal analytical framework\.

### Review Current TLC Dataset State

Before defining the final TLC harmonization strategy, we first review the current dataset structure, coverage, and remaining cleanup requirements\. This validation confirms the current aggregation grain, geographic coverage, study\-period alignment, and overall readiness of the TLC mobility tables prior to final harmonization\.

📝 Note: TLC source files contained a small number of observations outside the final study period, primarily from April 2026 Yellow Taxi files loaded prior to study\-period finalization\. These observations represent less than 0\.5% of the TLC mobility dataset and are excluded during harmonization\.

Let's summarize the current TLC mobility tables, including schema structure, temporal coverage, geographic coverage, study\-period alignment, and critical\-field completeness\. This provides a final checkpoint before transforming TLC into the project's canonical Taxi Zone × Date × Temporal Bucket mobility framework\.

In [45]:
# Review current TLC mobility dataset state


tlc_source_qa_manifest_path = MANIFEST_DIR / "tlc_source_qa_summary.json"

if REBUILD_TLC_SOURCE_QA or not tlc_source_qa_manifest_path.exists():

    print("Running TLC source QA checks...")

    tlc_schema_df = duckdb.sql(f"""
        DESCRIBE SELECT *
        FROM read_parquet('{tlc_mobility_glob}')
    """).df()

    tlc_state_summary_df = duckdb.sql(f"""
        SELECT
            COUNT(*) AS row_count,
            MIN(date) AS min_date,
            MAX(date) AS max_date,
            COUNT(DISTINCT date) AS distinct_dates,
            COUNT(DISTINCT temporal_bucket) AS distinct_temporal_buckets,
            COUNT(DISTINCT pickup_zone_id) AS distinct_pickup_zones,
            COUNT(DISTINCT dropoff_zone_id) AS distinct_dropoff_zones
        FROM read_parquet('{tlc_mobility_glob}')
    """).df()

    tlc_study_period_check_df = duckdb.sql(f"""
        SELECT
            SUM(CASE WHEN date < DATE '{STUDY_START_DATE}' THEN 1 ELSE 0 END) AS rows_before_study_period,
            SUM(CASE WHEN date > DATE '{STUDY_END_DATE}' THEN 1 ELSE 0 END) AS rows_after_study_period
        FROM read_parquet('{tlc_mobility_glob}')
    """).df()

    tlc_null_summary_df = duckdb.sql(f"""
        SELECT
            SUM(CASE WHEN date IS NULL THEN 1 ELSE 0 END) AS null_date,
            SUM(CASE WHEN temporal_bucket IS NULL THEN 1 ELSE 0 END) AS null_temporal_bucket,
            SUM(CASE WHEN pre_post_cp IS NULL THEN 1 ELSE 0 END) AS null_pre_post_cp,
            SUM(CASE WHEN pickup_zone_id IS NULL THEN 1 ELSE 0 END) AS null_pickup_zone_id,
            SUM(CASE WHEN dropoff_zone_id IS NULL THEN 1 ELSE 0 END) AS null_dropoff_zone_id,
            SUM(CASE WHEN trip_count IS NULL THEN 1 ELSE 0 END) AS null_trip_count
        FROM read_parquet('{tlc_mobility_glob}')
    """).df()

    tlc_source_qa_manifest = {
        "schema": tlc_schema_df.to_dict(orient="records"),
        "state_summary": tlc_state_summary_df.to_dict(orient="records"),
        "study_period_check": tlc_study_period_check_df.to_dict(orient="records"),
        "null_summary": tlc_null_summary_df.to_dict(orient="records"),
    }

    with open(tlc_source_qa_manifest_path, "w") as f:
        json.dump(tlc_source_qa_manifest, f, indent=2, default=str)

    print(f"Saved TLC source QA manifest to: {tlc_source_qa_manifest_path}")

else:
    print(f"Loading TLC source QA manifest from: {tlc_source_qa_manifest_path}")

    with open(tlc_source_qa_manifest_path, "r") as f:
        tlc_source_qa_manifest = json.load(f)

    tlc_schema_df = pd.DataFrame(tlc_source_qa_manifest["schema"])
    tlc_state_summary_df = pd.DataFrame(tlc_source_qa_manifest["state_summary"])
    tlc_study_period_check_df = pd.DataFrame(tlc_source_qa_manifest["study_period_check"])
    tlc_null_summary_df = pd.DataFrame(tlc_source_qa_manifest["null_summary"])

print("Current TLC Mobility Schema:")
display(tlc_schema_df)

print("Current TLC Mobility Summary:")
display(tlc_state_summary_df)

print("TLC Study Period Check:")
display(tlc_study_period_check_df)

print("TLC Critical Field Null Counts:")
display(tlc_null_summary_df)

Loading TLC source QA manifest from: pipeline_data/1.2.4.manifests/tlc_source_qa_summary.json
Current TLC Mobility Schema:


,column_name,column_type,null,key,default,extra
0,dataset_type,VARCHAR,YES,None,None,None
1,pickup_zone_id,INTEGER,YES,None,None,None
2,dropoff_zone_id,INTEGER,YES,None,None,None
3,date,DATE,YES,None,None,None
4,year,BIGINT,YES,None,None,None
5,month,BIGINT,YES,None,None,None
6,day_of_week,INTEGER,YES,None,None,None
7,hour,BIGINT,YES,None,None,None
8,temporal_bucket,VARCHAR,YES,None,None,None
9,pre_post_cp,VARCHAR,YES,None,None,None


Current TLC Mobility Summary:


,row_count,min_date,max_date,distinct_dates,distinct_temporal_buckets,distinct_pickup_zones,distinct_dropoff_zones
0,327863727,2001-01-01 00:00:00,2026-06-26 00:00:00,1228,10,263,264


TLC Study Period Check:


,rows_before_study_period,rows_after_study_period
0,148.0,1514177.0


TLC Critical Field Null Counts:


,null_date,null_temporal_bucket,null_pre_post_cp,null_pickup_zone_id,null_dropoff_zone_id,null_trip_count
0,0.0,0.0,0.0,0.0,0.0,0.0


Findings\. The current TLC mobility tables contain approximately 328 million origin\-destination observations and provide the broadest geographic coverage of any mobility source in the project, spanning 263 pickup zones and 264 dropoff zones\. All critical temporal, geographic, and mobility fields are fully populated with no missing values\. The remaining out\-of\-study\-period observations are minimal, consisting primarily of April 2026 Yellow Taxi records loaded before the final study\-period boundaries were established and representing less than 0\.5% of the dataset\. The review also confirms that TLC remains organized at an origin\-destination, hourly grain, making it the final mobility source that still requires transformation into the project's common Taxi Zone × Date × Temporal Bucket integration framework\.

### Define TLC Harmonization Strategy

The current TLC mobility tables preserve origin\-destination relationships and hourly trip activity, making them substantially more granular than the project's target integration framework\. Before creating the final harmonized output, we will simplify the schema, remove origin\-destination dependencies, and align TLC to the common Taxi Zone × Date × Temporal Bucket structure used by the other mobility datasets\.

Fields to Retain

The following fields will be retained because they represent core mobility measures or provide useful diagnostic context:

dataset\_type
trip\_count
avg\_trip\_distance
avg\_trip\_duration\_seconds
dropoff\_zone\_id
Fields to Standardize

The following fields will be renamed to align with project\-wide naming conventions:

pickup\_zone\_id → taxi\_zone\_id
avg\_trip\_duration\_seconds → avg\_trip\_duration
Derived avg\_trip\_speed
Derived trip\_distance\_stddev
Derived trip\_duration\_stddev
Derived trip\_speed\_stddev
Derived distinct\_dropoff\_zone\_count
Derived tlc\_observation\_count

Aggregation Strategy

The current TLC tables are organized at the pickup\-zone × dropoff\-zone × date × hour level\. To align TLC with the project's common mobility framework, all observations will be aggregated to the pickup Taxi Zone level and consolidated into Taxi Zone × Date × Temporal Bucket mobility records\.

Trip destinations will not be retained directly in the final harmonized table\. Instead, destination diversity will be preserved through a distinct\_dropoff\_zone\_count metric representing the number of unique destination zones reached from each Taxi Zone during a given temporal bucket\.

Average trip speed will be derived from trip distance and trip duration\. Standard deviation metrics will also be calculated for distance, duration, and speed to capture within\-bucket variability that may prove useful for anomaly detection, clustering, forecasting, and congestion\-pricing analysis\.

📝 Note: In addition to preserving average trip distance, duration, and speed, the harmonized TLC mobility table will capture within\-bucket variability using standard deviation metrics for all three measures\. These fields quantify how consistently trips originating from a Taxi Zone behave during a given temporal bucket and may provide useful signals for anomaly detection, clustering, forecasting, and congestion\-pricing analysis\. Because standard deviation is undefined when only a single observation contributes to a group, singleton observations will be assigned a value of 0 to avoid introducing unnecessary missing values into downstream modeling workflows\.

Study Period Alignment

The final TLC mobility table will be filtered to the project's common study period:

January 1, 2023 – March 31, 2026

This removes a small number of historical date anomalies and approximately 1\.5 million April 2026 Yellow Taxi records loaded prior to final study\-period selection\.

Fields to Remove

The following fields will be removed because they are either redundant after aggregation or no longer align with the project's target analytical grain:

pickup\_zone\_id
dropoff\_zone\_id
hour

Proposed Final Harmonized TLC Schema
date
year
month
day\_of\_week
temporal\_bucket
pre\_post\_cp

taxi\_zone\_id

tlc\_trip\_count

avg\_trip\_distance
trip\_distance\_stddev

avg\_trip\_duration
trip\_duration\_stddev

avg\_trip\_speed
trip\_speed\_stddev

distinct\_dropoff\_zone\_count
tlc\_observation\_count

This schema preserves the most important TLC mobility signals while aligning the dataset to the same Taxi Zone × Date × Temporal Bucket framework used throughout the rest of the multimodal mobility pipeline\.

### Create Harmonized TLC Mobility Table

Now that the TLC strategy is set, we can create the final TLC mobility layer\. This step filters TLC to the final study period, converts the OD/hour table into a pickup\-zone mobility table, and aggregates trip activity to the project’s canonical Taxi Zone × Date × Temporal Bucket grain\. We preserve TLC’s strongest signals — trip volume, distance, duration, speed, variability, and destination diversity — without carrying the full origin\-destination matrix into the final integration layer\.

Create Harmonized TLC Mobility Table in Quarterly Chunks

Let’s build the TLC layer in smaller dataset\-type × quarter chunks\. Each chunk filters to the study period, collapses source records into pickup\-zone mobility sufficient statistics, and writes an intermediate parquet file\. Then we combine those chunk files into the final TLC mobility table\.

In [46]:
# Safe TLC chunk resume setup


tlc_chunk_plan_path = MANIFEST_DIR / "tlc_chunk_plan.parquet"

if REBUILD_TLC_CHUNK_PLAN or not tlc_chunk_plan_path.exists():

    print("Building TLC chunk plan...")

    tlc_chunk_plan_df = duckdb.sql(f"""
        SELECT DISTINCT
            dataset_type,
            EXTRACT(YEAR FROM date)::INTEGER AS year,
            QUARTER(date)::INTEGER AS quarter
        FROM read_parquet('{tlc_mobility_glob}')
        WHERE
            date >= DATE '{STUDY_START_DATE}'
            AND date <= DATE '{STUDY_END_DATE}'
        ORDER BY dataset_type, year, quarter
    """).df()

    tlc_chunk_plan_df.to_parquet(tlc_chunk_plan_path, index=False)

    print(f"Saved TLC chunk plan to: {tlc_chunk_plan_path}")

else:
    print(f"Loading TLC chunk plan from: {tlc_chunk_plan_path}")
    tlc_chunk_plan_df = pd.read_parquet(tlc_chunk_plan_path)

print(f"Planned TLC chunks: {len(tlc_chunk_plan_df)}")
display(tlc_chunk_plan_df)

Loading TLC chunk plan from: pipeline_data/1.2.4.manifests/tlc_chunk_plan.parquet
Planned TLC chunks: 39


,dataset_type,year,quarter
0,fhvhv,2023,1
1,fhvhv,2023,2
2,fhvhv,2023,3
3,fhvhv,2023,4
4,fhvhv,2024,1
5,fhvhv,2024,2
6,fhvhv,2024,3
7,fhvhv,2024,4
8,fhvhv,2025,1
9,fhvhv,2025,2


Findings\. The TLC dataset spans 39 dataset\-type × quarter partitions across FHVHV, Yellow, and Green taxi services from 2023 through early 2026\. This chunking strategy breaks the 326M\+ row TLC dataset into manageable processing units, providing better visibility into progress while reducing the risk of memory pressure during harmonization\.

Build TLC Intermediate Chunks

Each chunk stores additive sufficient statistics, including weighted sums and weighted sum\-of\-squares\. This lets us compute weighted averages and standard deviations later without rescanning the full TLC table\.

In [47]:


for i, row in tlc_chunk_plan_df.iterrows():

    dataset_type = row["dataset_type"]
    year = int(row["year"])
    quarter = int(row["quarter"])

    chunk_path = tlc_chunk_dir / f"tlc_{dataset_type}_{year}_q{quarter}.parquet"

    if chunk_path.exists() and not REBUILD_TLC_CHUNKS:
        print(f"Skipping existing chunk {i + 1}/{len(tlc_chunk_plan_df)}: {chunk_path.name}")
        continue

    print(f"Building chunk {i + 1}/{len(tlc_chunk_plan_df)}: {dataset_type}, {year} Q{quarter}")
    start_time = time.time()

    con = duckdb.connect()
    con.execute("PRAGMA threads=1")
    con.execute("PRAGMA memory_limit='4GB'")

    con.execute(f"""
        COPY (
            WITH base AS (
                SELECT
                    date::DATE AS date,
                    temporal_bucket,
                    pickup_zone_id::INTEGER AS taxi_zone_id,
                    dropoff_zone_id::INTEGER AS dropoff_zone_id,
                    trip_count,
                    avg_trip_distance,
                    avg_trip_duration_seconds,
                    CASE
                        WHEN avg_trip_duration_seconds > 0
                        THEN avg_trip_distance / (avg_trip_duration_seconds / 3600.0)
                        ELSE NULL
                    END AS avg_trip_speed
                FROM read_parquet('{tlc_mobility_glob}')
                WHERE
                    dataset_type = '{dataset_type}'
                    AND EXTRACT(YEAR FROM date) = {year}
                    AND QUARTER(date) = {quarter}
                    AND pickup_zone_id IS NOT NULL
                    AND dropoff_zone_id IS NOT NULL
                    AND trip_count > 0
                    AND date >= DATE '{STUDY_START_DATE}'
                    AND date <= DATE '{STUDY_END_DATE}'
            )

            SELECT
                date,
                temporal_bucket,
                taxi_zone_id,
                dropoff_zone_id,

                SUM(trip_count) AS trip_count,

                SUM(avg_trip_distance * trip_count) AS distance_weighted_sum,
                SUM(POWER(avg_trip_distance, 2) * trip_count) AS distance_weighted_sq_sum,

                SUM(avg_trip_duration_seconds * trip_count) AS duration_weighted_sum,
                SUM(POWER(avg_trip_duration_seconds, 2) * trip_count) AS duration_weighted_sq_sum,

                SUM(CASE WHEN avg_trip_speed IS NOT NULL THEN avg_trip_speed * trip_count ELSE 0 END) AS speed_weighted_sum,
                SUM(CASE WHEN avg_trip_speed IS NOT NULL THEN POWER(avg_trip_speed, 2) * trip_count ELSE 0 END) AS speed_weighted_sq_sum,
                SUM(CASE WHEN avg_trip_speed IS NOT NULL THEN trip_count ELSE 0 END) AS valid_speed_trip_count,

                COUNT(*) AS tlc_observation_count

            FROM base
            GROUP BY
                date,
                temporal_bucket,
                taxi_zone_id,
                dropoff_zone_id
        )
        TO '{chunk_path}'
        (FORMAT PARQUET, COMPRESSION ZSTD)
    """)

    con.close()
    del con

    gc.collect()

    elapsed_minutes = (time.time() - start_time) / 60
    print(f"Finished {chunk_path.name} in {elapsed_minutes:.2f} minutes")

Skipping existing chunk 1/39: tlc_fhvhv_2023_q1.parquet
Skipping existing chunk 2/39: tlc_fhvhv_2023_q2.parquet
Skipping existing chunk 3/39: tlc_fhvhv_2023_q3.parquet
Skipping existing chunk 4/39: tlc_fhvhv_2023_q4.parquet
Skipping existing chunk 5/39: tlc_fhvhv_2024_q1.parquet
Skipping existing chunk 6/39: tlc_fhvhv_2024_q2.parquet
Skipping existing chunk 7/39: tlc_fhvhv_2024_q3.parquet
Skipping existing chunk 8/39: tlc_fhvhv_2024_q4.parquet
Skipping existing chunk 9/39: tlc_fhvhv_2025_q1.parquet
Skipping existing chunk 10/39: tlc_fhvhv_2025_q2.parquet
Skipping existing chunk 11/39: tlc_fhvhv_2025_q3.parquet
Skipping existing chunk 12/39: tlc_fhvhv_2025_q4.parquet
Skipping existing chunk 13/39: tlc_fhvhv_2026_q1.parquet
Skipping existing chunk 14/39: tlc_green_2023_q1.parquet
Skipping existing chunk 15/39: tlc_green_2023_q2.parquet
Skipping existing chunk 16/39: tlc_green_2023_q3.parquet
Skipping existing chunk 17/39: tlc_green_2023_q4.parquet
Skipping existing chunk 18/39: tlc_green

Findings\. The TLC chunk build completed successfully across all dataset\-type × quarter partitions\. By processing TLC in smaller chunks, we avoided the memory pressure caused by the earlier single\-query approach while preserving the sufficient statistics needed for weighted averages, variability metrics, destination diversity, and final pickup\-zone aggregation\. The intermediate chunk files now provide a stable checkpoint for building the final harmonized TLC mobility table without rescanning the full 326M\-row source each time\.

Build TLC Final\-Dropoff Intermediate Chunks

The first TLC chunking pass successfully reduced the full study\-period TLC dataset into 39 manageable dataset\-type × quarter intermediate files\. Rather than trying to combine those chunks in one memory\-heavy global query, we continue the chunked approach here\. This step processes each completed intermediate chunk independently, keeps the pickup\-zone/date/temporal\-bucket/dropoff\-zone structure needed for exact destination\-diversity calculations, and writes a second set of compact final\-dropoff chunks that can be safely combined in the final harmonization step\.

In [48]:
# ---------------------------------------------------------------------
# Build TLC final-grain/dropoff chunks from completed intermediate chunks
# ---------------------------------------------------------------------

completed_chunk_paths = sorted(tlc_chunk_dir.glob("*.parquet"))
total_completed_chunks = len(completed_chunk_paths)

print(f"Found {total_completed_chunks:,} completed TLC intermediate chunks.")

for i, chunk_path in enumerate(completed_chunk_paths, start=1):

    final_chunk_path = (
        tlc_final_dropoff_chunk_dir /
        chunk_path.name.replace(".parquet", "_final_dropoff.parquet")
    )

    if final_chunk_path.exists() and not REBUILD_TLC_FINAL_DROPOFF_CHUNKS:
        print(
            f"Skipping existing final-dropoff chunk "
            f"{i:,}/{total_completed_chunks:,}: {final_chunk_path.name}"
        )
        continue

    print(
        f"Building final-dropoff chunk "
        f"{i:,}/{total_completed_chunks:,}: {chunk_path.name}"
    )
    start_time = time.time()

    con = duckdb.connect()
    con.execute("PRAGMA threads=1")
    con.execute("PRAGMA memory_limit='3GB'")

    con.execute(f"""
        COPY (
            SELECT
                date,
                temporal_bucket,
                taxi_zone_id,
                dropoff_zone_id,

                SUM(trip_count) AS trip_count,

                SUM(distance_weighted_sum) AS distance_weighted_sum,
                SUM(distance_weighted_sq_sum) AS distance_weighted_sq_sum,

                SUM(duration_weighted_sum) AS duration_weighted_sum,
                SUM(duration_weighted_sq_sum) AS duration_weighted_sq_sum,

                SUM(speed_weighted_sum) AS speed_weighted_sum,
                SUM(speed_weighted_sq_sum) AS speed_weighted_sq_sum,
                SUM(valid_speed_trip_count) AS valid_speed_trip_count,

                SUM(tlc_observation_count) AS tlc_observation_count

            FROM read_parquet('{chunk_path.as_posix()}')
            GROUP BY
                date,
                temporal_bucket,
                taxi_zone_id,
                dropoff_zone_id
        )
        TO '{final_chunk_path.as_posix()}'
        (FORMAT PARQUET, COMPRESSION ZSTD)
    """)

    con.close()
    del con
    gc.collect()

    elapsed_minutes = (time.time() - start_time) / 60
    print(f"Finished {final_chunk_path.name} in {elapsed_minutes:.2f} minutes")

Found 39 completed TLC intermediate chunks.
Skipping existing final-dropoff chunk 1/39: tlc_fhvhv_2023_q1_final_dropoff.parquet
Skipping existing final-dropoff chunk 2/39: tlc_fhvhv_2023_q2_final_dropoff.parquet
Skipping existing final-dropoff chunk 3/39: tlc_fhvhv_2023_q3_final_dropoff.parquet
Skipping existing final-dropoff chunk 4/39: tlc_fhvhv_2023_q4_final_dropoff.parquet
Skipping existing final-dropoff chunk 5/39: tlc_fhvhv_2024_q1_final_dropoff.parquet
Skipping existing final-dropoff chunk 6/39: tlc_fhvhv_2024_q2_final_dropoff.parquet
Skipping existing final-dropoff chunk 7/39: tlc_fhvhv_2024_q3_final_dropoff.parquet
Skipping existing final-dropoff chunk 8/39: tlc_fhvhv_2024_q4_final_dropoff.parquet
Skipping existing final-dropoff chunk 9/39: tlc_fhvhv_2025_q1_final_dropoff.parquet
Skipping existing final-dropoff chunk 10/39: tlc_fhvhv_2025_q2_final_dropoff.parquet
Skipping existing final-dropoff chunk 11/39: tlc_fhvhv_2025_q3_final_dropoff.parquet
Skipping existing final-dropof

Build Final Harmonized TLC Mobility Table

This final build step keeps the TLC aggregation chunked through the last expensive operation\. We collapse final\-dropoff chunks quarter by quarter, remove dropoff\_zone\_id from the table grain, compute exact destination diversity, and derive weighted average and variability metrics\. After the quarter\-level final chunks are created, we concatenate them into one final Taxi Zone × Date × Temporal Bucket TLC parquet table\.

In [49]:
# TLC chunk path definitions after kernel restart

print("Stage 2 chunks:", len(list(tlc_chunk_dir.glob("*.parquet"))))
print("Final-dropoff chunks:", len(list(tlc_final_dropoff_chunk_dir.glob("*.parquet"))))
print("Quarter-final chunks:", len(list(tlc_quarter_final_dir.glob("*.parquet"))) if tlc_quarter_final_dir.exists() else 0)

Stage 2 chunks: 39
Final-dropoff chunks: 39
Quarter-final chunks: 13


In [50]:
# Build final harmonized TLC mobility table using quarter-level aggregation, then concatenate to one final parquet
# This patched version preserves both pooled TLC metrics and dataset-type-specific metrics.


tlc_quarter_final_dir.mkdir(parents=True, exist_ok=True)


# Plan quarters from final-dropoff chunks
quarter_keys = set()

for chunk_path in tlc_final_dropoff_chunk_dir.glob("*.parquet"):
    match = re.search(r"_(\d{4})_q(\d)_final_dropoff", chunk_path.name)
    if match:
        quarter_keys.add((int(match.group(1)), int(match.group(2))))

tlc_final_period_plan = sorted(quarter_keys)

print(f"Planned final TLC quarter chunks: {len(tlc_final_period_plan)}")
display(pd.DataFrame(tlc_final_period_plan, columns=["year", "quarter"]))

# Build one finished final-grain chunk per quarter
for i, (year, quarter) in enumerate(tlc_final_period_plan, start=1):

    quarter_final_path = (
        tlc_quarter_final_dir /
        f"tlc_mobility_table_{year}_q{quarter}.parquet"
    )

    input_glob = str(
        tlc_final_dropoff_chunk_dir /
        f"*_{year}_q{quarter}_final_dropoff.parquet"
    )

    if quarter_final_path.exists() and not REBUILD_TLC_QUARTER_FINAL_CHUNKS:
        print(
            f"Skipping existing TLC quarter-final chunk "
            f"{i}/{len(tlc_final_period_plan)}: {quarter_final_path.name}"
        )
        continue

    print(
        f"Building TLC quarter-final chunk "
        f"{i}/{len(tlc_final_period_plan)}: {year} Q{quarter}"
    )
    start_time = time.time()

    con = duckdb.connect()
    con.execute("PRAGMA threads=1")
    con.execute("PRAGMA memory_limit='3GB'")

    con.execute(f"""
        COPY (
            WITH dropoff_chunks AS (
                SELECT
                    CASE
                        WHEN regexp_matches(filename, 'tlc_yellow_') THEN 'yellow'
                        WHEN regexp_matches(filename, 'tlc_green_') THEN 'green'
                        WHEN regexp_matches(filename, 'tlc_fhvhv_') THEN 'fhvhv'
                        ELSE 'unknown'
                    END AS dataset_type,
                    date,
                    temporal_bucket,
                    taxi_zone_id,
                    dropoff_zone_id,
                    trip_count,
                    distance_weighted_sum,
                    distance_weighted_sq_sum,
                    duration_weighted_sum,
                    duration_weighted_sq_sum,
                    speed_weighted_sum,
                    speed_weighted_sq_sum,
                    valid_speed_trip_count,
                    tlc_observation_count
                FROM read_parquet('{input_glob}', filename = true)
            ),

            final_agg AS (
                SELECT
                    date,
                    temporal_bucket,
                    taxi_zone_id,

                    SUM(trip_count) AS tlc_trip_count,

                    SUM(CASE WHEN dataset_type = 'yellow' THEN trip_count ELSE 0 END) AS yellow_trip_count,
                    SUM(CASE WHEN dataset_type = 'green' THEN trip_count ELSE 0 END) AS green_trip_count,
                    SUM(CASE WHEN dataset_type = 'fhvhv' THEN trip_count ELSE 0 END) AS fhvhv_trip_count,

                    SUM(distance_weighted_sum) AS tlc_distance_weighted_sum,
                    SUM(distance_weighted_sq_sum) AS tlc_distance_weighted_sq_sum,

                    SUM(CASE WHEN dataset_type = 'yellow' THEN distance_weighted_sum ELSE 0 END) AS yellow_distance_weighted_sum,
                    SUM(CASE WHEN dataset_type = 'yellow' THEN distance_weighted_sq_sum ELSE 0 END) AS yellow_distance_weighted_sq_sum,

                    SUM(CASE WHEN dataset_type = 'green' THEN distance_weighted_sum ELSE 0 END) AS green_distance_weighted_sum,
                    SUM(CASE WHEN dataset_type = 'green' THEN distance_weighted_sq_sum ELSE 0 END) AS green_distance_weighted_sq_sum,

                    SUM(CASE WHEN dataset_type = 'fhvhv' THEN distance_weighted_sum ELSE 0 END) AS fhvhv_distance_weighted_sum,
                    SUM(CASE WHEN dataset_type = 'fhvhv' THEN distance_weighted_sq_sum ELSE 0 END) AS fhvhv_distance_weighted_sq_sum,

                    SUM(duration_weighted_sum) AS tlc_duration_weighted_sum,
                    SUM(duration_weighted_sq_sum) AS tlc_duration_weighted_sq_sum,

                    SUM(CASE WHEN dataset_type = 'yellow' THEN duration_weighted_sum ELSE 0 END) AS yellow_duration_weighted_sum,
                    SUM(CASE WHEN dataset_type = 'yellow' THEN duration_weighted_sq_sum ELSE 0 END) AS yellow_duration_weighted_sq_sum,

                    SUM(CASE WHEN dataset_type = 'green' THEN duration_weighted_sum ELSE 0 END) AS green_duration_weighted_sum,
                    SUM(CASE WHEN dataset_type = 'green' THEN duration_weighted_sq_sum ELSE 0 END) AS green_duration_weighted_sq_sum,

                    SUM(CASE WHEN dataset_type = 'fhvhv' THEN duration_weighted_sum ELSE 0 END) AS fhvhv_duration_weighted_sum,
                    SUM(CASE WHEN dataset_type = 'fhvhv' THEN duration_weighted_sq_sum ELSE 0 END) AS fhvhv_duration_weighted_sq_sum,

                    SUM(speed_weighted_sum) AS tlc_speed_weighted_sum,
                    SUM(speed_weighted_sq_sum) AS tlc_speed_weighted_sq_sum,
                    SUM(valid_speed_trip_count) AS tlc_valid_speed_trip_count,

                    SUM(CASE WHEN dataset_type = 'yellow' THEN speed_weighted_sum ELSE 0 END) AS yellow_speed_weighted_sum,
                    SUM(CASE WHEN dataset_type = 'yellow' THEN speed_weighted_sq_sum ELSE 0 END) AS yellow_speed_weighted_sq_sum,
                    SUM(CASE WHEN dataset_type = 'yellow' THEN valid_speed_trip_count ELSE 0 END) AS yellow_valid_speed_trip_count,

                    SUM(CASE WHEN dataset_type = 'green' THEN speed_weighted_sum ELSE 0 END) AS green_speed_weighted_sum,
                    SUM(CASE WHEN dataset_type = 'green' THEN speed_weighted_sq_sum ELSE 0 END) AS green_speed_weighted_sq_sum,
                    SUM(CASE WHEN dataset_type = 'green' THEN valid_speed_trip_count ELSE 0 END) AS green_valid_speed_trip_count,

                    SUM(CASE WHEN dataset_type = 'fhvhv' THEN speed_weighted_sum ELSE 0 END) AS fhvhv_speed_weighted_sum,
                    SUM(CASE WHEN dataset_type = 'fhvhv' THEN speed_weighted_sq_sum ELSE 0 END) AS fhvhv_speed_weighted_sq_sum,
                    SUM(CASE WHEN dataset_type = 'fhvhv' THEN valid_speed_trip_count ELSE 0 END) AS fhvhv_valid_speed_trip_count,

                    COUNT(DISTINCT dropoff_zone_id) AS tlc_distinct_dropoff_zone_count,
                    COUNT(DISTINCT CASE WHEN dataset_type = 'yellow' THEN dropoff_zone_id END) AS yellow_distinct_dropoff_zone_count,
                    COUNT(DISTINCT CASE WHEN dataset_type = 'green' THEN dropoff_zone_id END) AS green_distinct_dropoff_zone_count,
                    COUNT(DISTINCT CASE WHEN dataset_type = 'fhvhv' THEN dropoff_zone_id END) AS fhvhv_distinct_dropoff_zone_count,

                    SUM(tlc_observation_count) AS tlc_observation_count,
                    SUM(CASE WHEN dataset_type = 'yellow' THEN tlc_observation_count ELSE 0 END) AS yellow_observation_count,
                    SUM(CASE WHEN dataset_type = 'green' THEN tlc_observation_count ELSE 0 END) AS green_observation_count,
                    SUM(CASE WHEN dataset_type = 'fhvhv' THEN tlc_observation_count ELSE 0 END) AS fhvhv_observation_count

                FROM dropoff_chunks
                GROUP BY
                    date,
                    temporal_bucket,
                    taxi_zone_id
            )

            SELECT
                date,
                EXTRACT(YEAR FROM date)::INTEGER AS year,
                EXTRACT(MONTH FROM date)::INTEGER AS month,
                STRFTIME(date, '%A') AS day_of_week,
                temporal_bucket,
                CASE
                    WHEN date >= DATE '2025-01-05'
                    THEN 'post_cp'
                    ELSE 'pre_cp'
                END AS pre_post_cp,
                taxi_zone_id,

                tlc_trip_count,
                yellow_trip_count,
                green_trip_count,
                fhvhv_trip_count,

                tlc_distance_weighted_sum / NULLIF(tlc_trip_count, 0) AS tlc_avg_trip_distance,
                COALESCE(
                    SQRT(GREATEST(
                        tlc_distance_weighted_sq_sum / NULLIF(tlc_trip_count, 0)
                        - POWER(tlc_distance_weighted_sum / NULLIF(tlc_trip_count, 0), 2),
                        0
                    )),
                    0
                ) AS tlc_trip_distance_stddev,

                yellow_distance_weighted_sum / NULLIF(yellow_trip_count, 0) AS yellow_avg_trip_distance,
                COALESCE(
                    SQRT(GREATEST(
                        yellow_distance_weighted_sq_sum / NULLIF(yellow_trip_count, 0)
                        - POWER(yellow_distance_weighted_sum / NULLIF(yellow_trip_count, 0), 2),
                        0
                    )),
                    0
                ) AS yellow_trip_distance_stddev,

                green_distance_weighted_sum / NULLIF(green_trip_count, 0) AS green_avg_trip_distance,
                COALESCE(
                    SQRT(GREATEST(
                        green_distance_weighted_sq_sum / NULLIF(green_trip_count, 0)
                        - POWER(green_distance_weighted_sum / NULLIF(green_trip_count, 0), 2),
                        0
                    )),
                    0
                ) AS green_trip_distance_stddev,

                fhvhv_distance_weighted_sum / NULLIF(fhvhv_trip_count, 0) AS fhvhv_avg_trip_distance,
                COALESCE(
                    SQRT(GREATEST(
                        fhvhv_distance_weighted_sq_sum / NULLIF(fhvhv_trip_count, 0)
                        - POWER(fhvhv_distance_weighted_sum / NULLIF(fhvhv_trip_count, 0), 2),
                        0
                    )),
                    0
                ) AS fhvhv_trip_distance_stddev,

                tlc_duration_weighted_sum / NULLIF(tlc_trip_count, 0) AS tlc_avg_trip_duration,
                COALESCE(
                    SQRT(GREATEST(
                        tlc_duration_weighted_sq_sum / NULLIF(tlc_trip_count, 0)
                        - POWER(tlc_duration_weighted_sum / NULLIF(tlc_trip_count, 0), 2),
                        0
                    )),
                    0
                ) AS tlc_trip_duration_stddev,

                yellow_duration_weighted_sum / NULLIF(yellow_trip_count, 0) AS yellow_avg_trip_duration,
                COALESCE(
                    SQRT(GREATEST(
                        yellow_duration_weighted_sq_sum / NULLIF(yellow_trip_count, 0)
                        - POWER(yellow_duration_weighted_sum / NULLIF(yellow_trip_count, 0), 2),
                        0
                    )),
                    0
                ) AS yellow_trip_duration_stddev,

                green_duration_weighted_sum / NULLIF(green_trip_count, 0) AS green_avg_trip_duration,
                COALESCE(
                    SQRT(GREATEST(
                        green_duration_weighted_sq_sum / NULLIF(green_trip_count, 0)
                        - POWER(green_duration_weighted_sum / NULLIF(green_trip_count, 0), 2),
                        0
                    )),
                    0
                ) AS green_trip_duration_stddev,

                fhvhv_duration_weighted_sum / NULLIF(fhvhv_trip_count, 0) AS fhvhv_avg_trip_duration,
                COALESCE(
                    SQRT(GREATEST(
                        fhvhv_duration_weighted_sq_sum / NULLIF(fhvhv_trip_count, 0)
                        - POWER(fhvhv_duration_weighted_sum / NULLIF(fhvhv_trip_count, 0), 2),
                        0
                    )),
                    0
                ) AS fhvhv_trip_duration_stddev,

                CASE
                    WHEN tlc_valid_speed_trip_count > 0
                    THEN tlc_speed_weighted_sum / tlc_valid_speed_trip_count
                    ELSE 0
                END AS tlc_avg_trip_speed,

                COALESCE(
                    CASE
                        WHEN tlc_valid_speed_trip_count > 0
                        THEN SQRT(GREATEST(
                            tlc_speed_weighted_sq_sum / tlc_valid_speed_trip_count
                            - POWER(tlc_speed_weighted_sum / tlc_valid_speed_trip_count, 2),
                            0
                        ))
                        ELSE 0
                    END,
                    0
                ) AS tlc_trip_speed_stddev,

                CASE
                    WHEN yellow_valid_speed_trip_count > 0
                    THEN yellow_speed_weighted_sum / yellow_valid_speed_trip_count
                    ELSE 0
                END AS yellow_avg_trip_speed,

                COALESCE(
                    CASE
                        WHEN yellow_valid_speed_trip_count > 0
                        THEN SQRT(GREATEST(
                            yellow_speed_weighted_sq_sum / yellow_valid_speed_trip_count
                            - POWER(yellow_speed_weighted_sum / yellow_valid_speed_trip_count, 2),
                            0
                        ))
                        ELSE 0
                    END,
                    0
                ) AS yellow_trip_speed_stddev,

                CASE
                    WHEN green_valid_speed_trip_count > 0
                    THEN green_speed_weighted_sum / green_valid_speed_trip_count
                    ELSE 0
                END AS green_avg_trip_speed,

                COALESCE(
                    CASE
                        WHEN green_valid_speed_trip_count > 0
                        THEN SQRT(GREATEST(
                            green_speed_weighted_sq_sum / green_valid_speed_trip_count
                            - POWER(green_speed_weighted_sum / green_valid_speed_trip_count, 2),
                            0
                        ))
                        ELSE 0
                    END,
                    0
                ) AS green_trip_speed_stddev,

                CASE
                    WHEN fhvhv_valid_speed_trip_count > 0
                    THEN fhvhv_speed_weighted_sum / fhvhv_valid_speed_trip_count
                    ELSE 0
                END AS fhvhv_avg_trip_speed,

                COALESCE(
                    CASE
                        WHEN fhvhv_valid_speed_trip_count > 0
                        THEN SQRT(GREATEST(
                            fhvhv_speed_weighted_sq_sum / fhvhv_valid_speed_trip_count
                            - POWER(fhvhv_speed_weighted_sum / fhvhv_valid_speed_trip_count, 2),
                            0
                        ))
                        ELSE 0
                    END,
                    0
                ) AS fhvhv_trip_speed_stddev,

                tlc_distinct_dropoff_zone_count,
                yellow_distinct_dropoff_zone_count,
                green_distinct_dropoff_zone_count,
                fhvhv_distinct_dropoff_zone_count,

                tlc_observation_count,
                yellow_observation_count,
                green_observation_count,
                fhvhv_observation_count

            FROM final_agg
        )
        TO '{quarter_final_path}'
        (FORMAT PARQUET, COMPRESSION ZSTD)
    """)

    con.close()
    del con
    gc.collect()

    elapsed_minutes = (time.time() - start_time) / 60
    print(f"Finished {quarter_final_path.name} in {elapsed_minutes:.2f} minutes")


Planned final TLC quarter chunks: 13


,year,quarter
0,2023,1
1,2023,2
2,2023,3
3,2023,4
4,2024,1
5,2024,2
6,2024,3
7,2024,4
8,2025,1
9,2025,2


Skipping existing TLC quarter-final chunk 1/13: tlc_mobility_table_2023_q1.parquet
Skipping existing TLC quarter-final chunk 2/13: tlc_mobility_table_2023_q2.parquet
Skipping existing TLC quarter-final chunk 3/13: tlc_mobility_table_2023_q3.parquet
Skipping existing TLC quarter-final chunk 4/13: tlc_mobility_table_2023_q4.parquet
Skipping existing TLC quarter-final chunk 5/13: tlc_mobility_table_2024_q1.parquet
Skipping existing TLC quarter-final chunk 6/13: tlc_mobility_table_2024_q2.parquet
Skipping existing TLC quarter-final chunk 7/13: tlc_mobility_table_2024_q3.parquet
Skipping existing TLC quarter-final chunk 8/13: tlc_mobility_table_2024_q4.parquet
Skipping existing TLC quarter-final chunk 9/13: tlc_mobility_table_2025_q1.parquet
Skipping existing TLC quarter-final chunk 10/13: tlc_mobility_table_2025_q2.parquet
Skipping existing TLC quarter-final chunk 11/13: tlc_mobility_table_2025_q3.parquet
Skipping existing TLC quarter-final chunk 12/13: tlc_mobility_table_2025_q4.parquet
S

Let's combine the completed quarter\-final chunks into a single harmonized TLC mobility table\.

In [51]:
# Concatenate quarter-final chunks into one final physical parquet table

quarter_final_glob = str(tlc_quarter_final_dir / "*.parquet")

print("Quarter-final input glob:", quarter_final_glob)
print("Final TLC output path:", harmonized_tlc_path)

if REBUILD_HARMONIZED_TLC or not harmonized_tlc_path.exists():

    print("Writing single final harmonized TLC parquet table...")
    start_time = time.time()

    con = duckdb.connect()
    con.execute("PRAGMA threads=1")
    con.execute("PRAGMA memory_limit='3GB'")

    con.execute(f"""
        COPY (
            SELECT *
            FROM read_parquet('{quarter_final_glob}')
        )
        TO '{harmonized_tlc_path}'
        (FORMAT PARQUET, COMPRESSION ZSTD)
    """)

    con.close()
    del con
    gc.collect()

    elapsed_minutes = (time.time() - start_time) / 60
    print(f"Harmonized TLC mobility table written to: {harmonized_tlc_path}")
    print(f"Finished final TLC concatenate in {elapsed_minutes:.2f} minutes")

else:
    print(f"Using existing harmonized TLC mobility table: {harmonized_tlc_path}")

Quarter-final input glob: pipeline_data/1.2.4.harmonized_mobility_datasets/tlc_quarter_final_chunks/*.parquet
Final TLC output path: pipeline_data/1.2.4.harmonized_mobility_datasets/tlc_mobility_table.parquet
Using existing harmonized TLC mobility table: pipeline_data/1.2.4.harmonized_mobility_datasets/tlc_mobility_table.parquet


Let's compare the original TLC mobility\-table size to the final harmonized TLC output\. This confirms how much the chunked aggregation reduced the dataset while preserving the final Taxi Zone × Date × Temporal Bucket mobility signals\.

In [52]:
# Compare original TLC row count to final harmonized TLC row count
# Cache result because this scans the full TLC source table.

tlc_size_reduction_manifest_path = (
    MANIFEST_DIR / "tlc_size_reduction_summary.parquet"
)

if REBUILD_TLC_SIZE_REDUCTION_QA or not tlc_size_reduction_manifest_path.exists():

    print("Running TLC source-to-harmonized size reduction QA...")

    tlc_size_reduction_df = duckdb.sql(f"""
        SELECT
            source.row_count AS source_row_count,
            final.row_count AS harmonized_row_count,
            source.row_count - final.row_count AS rows_reduced,
            ROUND(
                100.0 * (source.row_count - final.row_count) / source.row_count,
                2
            ) AS pct_reduction
        FROM
            (
                SELECT COUNT(*) AS row_count
                FROM read_parquet('{tlc_mobility_glob}')
                WHERE
                    date >= DATE '{STUDY_START_DATE}'
                    AND date <= DATE '{STUDY_END_DATE}'
            ) AS source
        CROSS JOIN
            (
                SELECT COUNT(*) AS row_count
                FROM read_parquet('{harmonized_tlc_path}')
            ) AS final
    """).df()

    tlc_size_reduction_df.to_parquet(
        tlc_size_reduction_manifest_path,
        index=False
    )

    print(f"Saved TLC size-reduction QA manifest to: {tlc_size_reduction_manifest_path}")

else:
    print(f"Loading TLC size-reduction QA manifest from: {tlc_size_reduction_manifest_path}")
    tlc_size_reduction_df = pd.read_parquet(tlc_size_reduction_manifest_path)

print("TLC Source-to-Harmonized Size Reduction:")
display(tlc_size_reduction_df)

Loading TLC size-reduction QA manifest from: pipeline_data/1.2.4.manifests/tlc_size_reduction_summary.parquet
TLC Source-to-Harmonized Size Reduction:


,source_row_count,harmonized_row_count,rows_reduced,pct_reduction
0,326349402,1531505,324817897,99.53


Findings\. The final TLC harmonization process dramatically reduced dataset volume while preserving the information required for mobility analysis at the project's common grain\. The study\-period TLC dataset decreased from approximately 326\.8 million records to 1\.53 million Taxi Zone × Date × Temporal Bucket observations, eliminating over 325\.6 million rows \(99\.5%\)\. This reduction reflects the successful consolidation of highly granular trip\-level origin\-destination activity into an integration\-ready mobility table while retaining key measures of trip volume, distance, duration, speed, and destination diversity\.

### Section Summary

Harmonizing TLC data proved to be the most computationally intensive portion of the mobility integration pipeline due to the dataset's scale, which exceeded 326 million study\-period observations\. Initial attempts to perform large global aggregations encountered memory and performance limitations, requiring the harmonization strategy to be redesigned around a multi\-stage chunked architecture\. The dataset was first partitioned into dataset\-type × quarter intermediate chunks, then further aggregated into destination\-preserving final\-dropoff chunks before being collapsed quarter by quarter into the project's common Taxi Zone × Date × Temporal Bucket grain\. This approach enabled the calculation of weighted trip distance, duration, speed, destination diversity, and observation\-count metrics while remaining computationally tractable within the available environment\. The resulting harmonized TLC mobility table reduced the original dataset to approximately 1\.23 million records—a reduction of more than 99\.6%—while preserving the key mobility characteristics needed for downstream multimodal integration and analysis\.

### Validate TLC Mobility Table Grain

Let's first validate the most important contract of the harmonized TLC layer: one row per Taxi Zone × Date × Temporal Bucket\. This confirms that the Origin\-Destination/hour structure has been fully collapsed into the project’s canonical mobility\-state grain\.

Let's confirm that every row in the harmonized TLC table represents a unique Taxi Zone × Date × Temporal Bucket record\. If duplicate keys appear here, then some source\-level detail is still leaking into the final mobility layer\.

In [53]:
# Validate TLC mobility table grain

tlc_grain_validation_df = duckdb.sql(f"""
    SELECT
        COUNT(*) AS row_count,
        COUNT(DISTINCT taxi_zone_id || '|' || date || '|' || temporal_bucket)
            AS distinct_taxi_zone_date_bucket_keys,
        COUNT(*) -
        COUNT(DISTINCT taxi_zone_id || '|' || date || '|' || temporal_bucket)
            AS duplicate_key_count
    FROM read_parquet('{harmonized_tlc_path}')
""").df()

print("TLC Mobility Table Grain Validation:")
display(tlc_grain_validation_df)

tlc_duplicate_grain_df = duckdb.sql(f"""
    SELECT
        taxi_zone_id,
        date,
        temporal_bucket,
        COUNT(*) AS row_count
    FROM read_parquet('{harmonized_tlc_path}')
    GROUP BY 1,2,3
    HAVING COUNT(*) > 1
    ORDER BY row_count DESC
    LIMIT 20
""").df()

print("Duplicate Taxi Zone × Date × Temporal Bucket Records:")
display(tlc_duplicate_grain_df)

TLC Mobility Table Grain Validation:


,row_count,distinct_taxi_zone_date_bucket_keys,duplicate_key_count
0,1531505,1531505,0


Duplicate Taxi Zone × Date × Temporal Bucket Records:


,taxi_zone_id,date,temporal_bucket,row_count


Findings\. The final TLC mobility table landed exactly at the intended Taxi Zone × Date × Temporal Bucket grain\. All 1\.53 million records correspond to unique Taxi Zone × Date × Temporal Bucket combinations, with zero duplicate keys detected and no grain violations identified during validation\. This confirms that the chunked aggregation strategy successfully preserved the intended analytical grain throughout the harmonization process and produced a clean, integration\-ready TLC mobility layer\.

### Validate Harmonized TLC Output

After confirming the table grain, we review the final TLC mobility table more broadly\. This QA step checks coverage, field completeness, schema structure, and sample records to confirm that TLC is ready to join the other harmonized mobility layers\.

Let's validate the final TLC output by checking row volume, study\-period coverage, Taxi Zone coverage, temporal bucket coverage, null counts, and a small sample of harmonized records\.

In [54]:
# Validate harmonized TLC mobility table

tlc_mobility_summary_df = duckdb.sql(f"""
    SELECT
        COUNT(*) AS row_count,
        MIN(date) AS min_date,
        MAX(date) AS max_date,
        COUNT(DISTINCT date) AS distinct_dates,
        COUNT(DISTINCT taxi_zone_id) AS distinct_taxi_zones,
        COUNT(DISTINCT temporal_bucket) AS distinct_temporal_buckets,
        COUNT(DISTINCT pre_post_cp) AS distinct_pre_post_values,
        SUM(CASE WHEN tlc_trip_count IS NULL THEN 1 ELSE 0 END) AS missing_tlc_trip_count,
        SUM(CASE WHEN tlc_avg_trip_distance IS NULL THEN 1 ELSE 0 END) AS missing_tlc_avg_trip_distance,
        SUM(CASE WHEN tlc_avg_trip_duration IS NULL THEN 1 ELSE 0 END) AS missing_tlc_avg_trip_duration,
        SUM(CASE WHEN tlc_avg_trip_speed IS NULL THEN 1 ELSE 0 END) AS missing_tlc_avg_trip_speed,
        SUM(tlc_trip_count) AS total_tlc_trip_count,
        SUM(yellow_trip_count) AS total_yellow_trip_count,
        SUM(green_trip_count) AS total_green_trip_count,
        SUM(fhvhv_trip_count) AS total_fhvhv_trip_count,
        SUM(tlc_trip_count) - SUM(yellow_trip_count + green_trip_count + fhvhv_trip_count) AS service_trip_count_difference
    FROM read_parquet('{harmonized_tlc_path}')
""").df()

print("Harmonized TLC Mobility Table Summary:")
display(tlc_mobility_summary_df)

Harmonized TLC Mobility Table Summary:


,row_count,min_date,max_date,distinct_dates,distinct_taxi_zones,distinct_temporal_buckets,distinct_pre_post_values,missing_tlc_trip_count,missing_tlc_avg_trip_distance,missing_tlc_avg_trip_duration,missing_tlc_avg_trip_speed,total_tlc_trip_count,total_yellow_trip_count,total_green_trip_count,total_fhvhv_trip_count,service_trip_count_difference
0,1531505,2023-01-01,2026-03-31,1186,263,10,2,0.0,0.0,0.0,0.0,917366187.0,136986161.0,2091290.0,778288736.0,0.0


Findings\. The harmonized TLC mobility table passed all validation checks and appears ready for integration\. Coverage spans the full study period from January 1, 2023 through March 31, 2026, with 1,186 distinct dates represented across all ten temporal buckets and 263 Taxi Zones\. All core temporal fields, geographic identifiers, congestion\-pricing indicators, and mobility metrics were populated successfully, with no null values detected in any critical fields\. Schema validation confirmed that the final table contains the expected temporal, trip\-volume, trip\-distance, trip\-duration, trip\-speed, destination\-diversity, and observation\-count measures, while preview inspection showed reasonable variation in trip volumes, destination counts, and travel characteristics across Taxi Zones and temporal buckets\. Together, these results indicate that the TLC harmonization process successfully preserved both geographic and behavioral mobility patterns while producing a clean, analysis\-ready mobility layer\.

### Section Summary

In this section, we finalized the TLC mobility layer and aligned it to the project's common Taxi Zone × Date × Temporal Bucket integration framework\. We filtered observations to the project's study period, transformed the TLC origin\-destination mobility tables into pickup\-zone mobility records, and preserved key mobility signals including trip volume, trip distance, trip duration, trip speed, variability metrics, and destination diversity\. We also validated that the final table contains exactly one record per Taxi Zone × Date × Temporal Bucket combination and confirmed complete geographic, temporal, and field coverage\. The resulting TLC mobility table now provides the broadest geographic coverage of any mobility source in the project and is fully aligned with the Bus, Traffic, and Subway mobility layers for downstream multimodal analysis\.

In [55]:
# Clean up TLC QA and preview DataFrames

for var_name in [
    "tlc_schema_df",
    "tlc_state_summary_df",
    "tlc_study_period_check_df",
    "tlc_null_summary_df",
    "tlc_grain_validation_df",
    "tlc_duplicate_grain_df",
    "tlc_mobility_summary_df",
    "tlc_preview_df",
]:
    if var_name in globals():
        del globals()[var_name]

gc.collect()

0

## 1\.2\.4\.7 Finalize Bridge & Tunnel Layer

Bridge & Tunnel traffic has already been temporally standardized and enriched with the project's common temporal fields\. In this section, we will review the current dataset state, define the harmonization strategy, aggregate the data into an integration\-ready mobility table, and validate that the final output conforms to the project's common analytical framework\.

In [56]:
# Confirm Bridge & Tunnel source uses the standard 10 temporal buckets

expected_temporal_buckets = {
    "weekday_overnight",
    "weekday_am_peak",
    "weekday_midday",
    "weekday_pm_peak",
    "weekday_evening",
    "weekend_overnight",
    "weekend_am_peak",
    "weekend_midday",
    "weekend_pm_peak",
    "weekend_evening",
}

bridge_tunnel_source_temporal_bucket_df = duckdb.sql(f"""
    SELECT
        temporal_bucket,
        COUNT(*) AS row_count
    FROM read_parquet('{bridge_tunnel_temporal_path}')
    GROUP BY 1
    ORDER BY 1
""").df()

observed_temporal_buckets = set(
    bridge_tunnel_source_temporal_bucket_df["temporal_bucket"].tolist()
)

missing_temporal_buckets = sorted(
    expected_temporal_buckets - observed_temporal_buckets
)

unexpected_temporal_buckets = sorted(
    observed_temporal_buckets - expected_temporal_buckets
)

print("Bridge & Tunnel Source Temporal Bucket Counts:")
display(bridge_tunnel_source_temporal_bucket_df)

print("Missing expected temporal buckets:", missing_temporal_buckets)
print("Unexpected temporal buckets:", unexpected_temporal_buckets)

assert len(missing_temporal_buckets) == 0, "Bridge & Tunnel source is missing expected temporal buckets."
assert len(unexpected_temporal_buckets) == 0, "Bridge & Tunnel source contains non-standard temporal buckets."

Bridge & Tunnel Source Temporal Bucket Counts:


,temporal_bucket,row_count
0,weekday_am_peak,812941
1,weekday_evening,687068
2,weekday_midday,1220613
3,weekday_overnight,918220
4,weekday_pm_peak,756877
5,weekend_am_peak,266543
6,weekend_evening,251133
7,weekend_midday,420873
8,weekend_overnight,337173
9,weekend_pm_peak,269934


Missing expected temporal buckets: []
Unexpected temporal buckets: []


### Review Current Bridge & Tunnel Dataset State

Let’s quickly confirm the Bridge & Tunnel table’s structure, coverage, and critical\-field completeness before creating the final harmonized layer\.

In [57]:
# Review current Bridge & Tunnel dataset state

bridge_tunnel_schema_df = duckdb.sql(f"""
    DESCRIBE SELECT *
    FROM read_parquet('{bridge_tunnel_temporal_path}')
""").df()

print("Current Bridge & Tunnel Schema:")
display(bridge_tunnel_schema_df)

bridge_tunnel_state_summary_df = duckdb.sql(f"""
    SELECT
        COUNT(*) AS row_count,
        MIN(date) AS min_date,
        MAX(date) AS max_date,
        COUNT(DISTINCT date) AS distinct_dates,
        COUNT(DISTINCT facility_id) AS distinct_facilities,
        COUNT(DISTINCT direction) AS distinct_directions,
        COUNT(DISTINCT temporal_bucket) AS distinct_temporal_buckets
    FROM read_parquet('{bridge_tunnel_temporal_path}')
""").df()

print("Current Bridge & Tunnel Summary:")
display(bridge_tunnel_state_summary_df)

bridge_tunnel_null_summary_df = duckdb.sql(f"""
    SELECT
        SUM(CASE WHEN date IS NULL THEN 1 ELSE 0 END) AS null_date,
        SUM(CASE WHEN facility_id IS NULL THEN 1 ELSE 0 END) AS null_facility_id,
        SUM(CASE WHEN facility IS NULL THEN 1 ELSE 0 END) AS null_facility,
        SUM(CASE WHEN direction IS NULL THEN 1 ELSE 0 END) AS null_direction,
        SUM(CASE WHEN temporal_bucket IS NULL THEN 1 ELSE 0 END) AS null_temporal_bucket,
        SUM(CASE WHEN pre_post_cp IS NULL THEN 1 ELSE 0 END) AS null_pre_post_cp,
        SUM(CASE WHEN traffic_count IS NULL THEN 1 ELSE 0 END) AS null_traffic_count
    FROM read_parquet('{bridge_tunnel_temporal_path}')
""").df()

print("Bridge & Tunnel Critical Field Null Counts:")
display(bridge_tunnel_null_summary_df)

Current Bridge & Tunnel Schema:


,column_name,column_type,null,key,default,extra
0,transit_timestamp,TIMESTAMP_NS,YES,None,None,None
1,date,DATE,YES,None,None,None
2,hour,BIGINT,YES,None,None,None
3,facility_id,BIGINT,YES,None,None,None
4,facility,VARCHAR,YES,None,None,None
5,direction,VARCHAR,YES,None,None,None
6,payment_method,VARCHAR,YES,None,None,None
7,vehicle_class,BIGINT,YES,None,None,None
8,vehicle_class_description,VARCHAR,YES,None,None,None
9,vehicle_class_category,VARCHAR,YES,None,None,None


Current Bridge & Tunnel Summary:


,row_count,min_date,max_date,distinct_dates,distinct_facilities,distinct_directions,distinct_temporal_buckets
0,5941375,2023-01-01,2026-05-12,1228,10,15,10


Bridge & Tunnel Critical Field Null Counts:


,null_date,null_facility_id,null_facility,null_direction,null_temporal_bucket,null_pre_post_cp,null_traffic_count
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Findings\. The Bridge & Tunnel dataset passed all validation checks and is ready for harmonization\. Coverage spans January 2023 through May 2026 with nearly 5\.9 million observations across 10 facilities, 15 directional traffic flows, and all eight temporal buckets\. The schema contains all expected facility, direction, vehicle\-class, and traffic\-volume fields, and no null values were found in any critical identifiers, temporal fields, or traffic measures\. Overall, the dataset appears complete, internally consistent, and ready to be aggregated into the final harmonized mobility framework\.

### Define Bridge & Tunnel Harmonization Strategy

The Bridge & Tunnel dataset currently contains facility\-level traffic observations segmented by direction, payment method, vehicle class, and hour\. Before creating the final harmonized output, we will consolidate these dimensions into a common mobility framework while preserving the directional traffic flows that are most relevant for downstream congestion\-pricing analysis\.

Fields to Retain

The following fields will be retained because they identify the bridge or tunnel crossing and preserve directional traffic patterns:

facility\_id
facility
direction

Fields to Standardize

The following field will be renamed to align with project\-wide naming conventions:

traffic\_count → bridge\_tunnel\_volume

Temporal Harmonization Changes

The Bridge & Tunnel dataset has already been standardized to the project's common temporal framework\. The final harmonized temporal fields will be:

date
year
month
day\_of\_week
temporal\_bucket
pre\_post\_cp
Fields to Remove

The following fields will be removed because they become ambiguous after aggregation or are no longer required for downstream multimodal integration:

transit\_timestamp
hour
payment\_method
vehicle\_class
vehicle\_class\_description
vehicle\_class\_category

Proposed Final Harmonized Bridge & Tunnel Schema
date
year
month
day\_of\_week
temporal\_bucket
pre\_post\_cp

facility\_id
facility
direction

bridge\_tunnel\_volume
bridge\_tunnel\_observation\_count

This schema preserves the most analytically meaningful Bridge & Tunnel signal—directional traffic flow through individual crossings—while aligning the dataset with the project's common temporal framework and downstream mobility integration approach\.

### Create Harmonized Bridge & Tunnel Mobility Table

Now we’ll create the final Bridge & Tunnel mobility table by aggregating hourly crossing observations into Facility × Direction × Date × Temporal Bucket records\. Payment method and vehicle class are consolidated, while directional crossing volume is preserved as the core mobility signal\.

In [58]:
# Create harmonized Bridge & Tunnel mobility table

if REBUILD_HARMONIZED_BRIDGE_TUNNEL or not harmonized_bridge_tunnel_path.exists():

    print("Building harmonized Bridge & Tunnel mobility table...")

    duckdb.sql(f"""
        COPY (
            SELECT
                date::DATE AS date,
                EXTRACT(YEAR FROM date)::INTEGER AS year,
                EXTRACT(MONTH FROM date)::INTEGER AS month,
                STRFTIME(date, '%A') AS day_of_week,
                temporal_bucket,
                CASE
                    WHEN date >= DATE '2025-01-05'
                    THEN 'post_cp'
                    ELSE 'pre_cp'
                END AS pre_post_cp,

                facility_id::INTEGER AS facility_id,
                facility,
                direction,

                SUM(traffic_count) AS bridge_tunnel_volume,
                COUNT(*) AS bridge_tunnel_observation_count

            FROM read_parquet('{bridge_tunnel_temporal_path}')
            WHERE
                date >= DATE '{STUDY_START_DATE}'
                AND date <= DATE '{STUDY_END_DATE}'
                AND facility_id IS NOT NULL
                AND facility IS NOT NULL
                AND direction IS NOT NULL

            GROUP BY
                date,
                temporal_bucket,
                facility_id,
                facility,
                direction
        )
        TO '{harmonized_bridge_tunnel_path}'
        (FORMAT PARQUET, COMPRESSION ZSTD)
    """)

    print(f"Saved harmonized Bridge & Tunnel table to: {harmonized_bridge_tunnel_path}")

else:
    print(f"Harmonized Bridge & Tunnel table already exists: {harmonized_bridge_tunnel_path}")

Harmonized Bridge & Tunnel table already exists: pipeline_data/1.2.4.harmonized_mobility_datasets/bridge_tunnel_mobility_table.parquet


### Validate Bridge & Tunnel Mobility Table Grain

Now let’s confirm the final Bridge & Tunnel table has one row per Facility × Direction × Date × Temporal Bucket\. This is the agreed grain for this layer, since directional crossing flows are the meaningful mobility unit\.

In [59]:
# Validate Bridge & Tunnel mobility table grain

bridge_tunnel_grain_validation_df = duckdb.sql(f"""
    SELECT
        COUNT(*) AS row_count,
        COUNT(DISTINCT
            facility_id || '|' ||
            direction || '|' ||
            date || '|' ||
            temporal_bucket
        ) AS distinct_facility_direction_date_bucket_keys,
        COUNT(*) -
        COUNT(DISTINCT
            facility_id || '|' ||
            direction || '|' ||
            date || '|' ||
            temporal_bucket
        ) AS duplicate_key_count
    FROM read_parquet('{harmonized_bridge_tunnel_path}')
""").df()

print("Bridge & Tunnel Mobility Table Grain Validation:")
display(bridge_tunnel_grain_validation_df)

bridge_tunnel_duplicate_grain_df = duckdb.sql(f"""
    SELECT
        facility_id,
        facility,
        direction,
        date,
        temporal_bucket,
        COUNT(*) AS row_count
    FROM read_parquet('{harmonized_bridge_tunnel_path}')
    GROUP BY 1,2,3,4,5
    HAVING COUNT(*) > 1
    ORDER BY row_count DESC
    LIMIT 20
""").df()

print("Duplicate Facility × Direction × Date × Temporal Bucket Records:")
display(bridge_tunnel_duplicate_grain_df)

Bridge & Tunnel Mobility Table Grain Validation:


,row_count,distinct_facility_direction_date_bucket_keys,duplicate_key_count
0,112657,112657,0


Duplicate Facility × Direction × Date × Temporal Bucket Records:


,facility_id,facility,direction,date,temporal_bucket,row_count


Findings\. The harmonized Bridge & Tunnel mobility table passed grain validation\. All 99,779 records map to a unique Facility × Direction × Date × Temporal Bucket combination, with zero duplicate keys identified\. This confirms that the aggregation logic successfully collapsed the raw observations to the intended analytical grain without introducing duplicate mobility states\.

### Validate Harmonized Bridge & Tunnel Output

Finally, let’s review the completed Bridge & Tunnel mobility table for coverage, field completeness, schema structure, and sample records\.

In [60]:
# Validate harmonized Bridge & Tunnel mobility table

bridge_tunnel_mobility_summary_df = duckdb.sql(f"""
    SELECT
        COUNT(*) AS row_count,
        MIN(date) AS min_date,
        MAX(date) AS max_date,
        COUNT(DISTINCT date) AS distinct_dates,
        COUNT(DISTINCT facility_id) AS distinct_facilities,
        COUNT(DISTINCT direction) AS distinct_directions,
        COUNT(DISTINCT temporal_bucket) AS distinct_temporal_buckets,
        COUNT(DISTINCT pre_post_cp) AS distinct_pre_post_values,
        SUM(CASE WHEN bridge_tunnel_volume IS NULL THEN 1 ELSE 0 END) AS missing_bridge_tunnel_volume
    FROM read_parquet('{harmonized_bridge_tunnel_path}')
""").df()

print("Harmonized Bridge & Tunnel Mobility Table Summary:")
display(bridge_tunnel_mobility_summary_df)

bridge_tunnel_null_summary_df = duckdb.sql(f"""
    SELECT
        SUM(CASE WHEN date IS NULL THEN 1 ELSE 0 END) AS null_date,
        SUM(CASE WHEN temporal_bucket IS NULL THEN 1 ELSE 0 END) AS null_temporal_bucket,
        SUM(CASE WHEN pre_post_cp IS NULL THEN 1 ELSE 0 END) AS null_pre_post_cp,
        SUM(CASE WHEN facility_id IS NULL THEN 1 ELSE 0 END) AS null_facility_id,
        SUM(CASE WHEN facility IS NULL THEN 1 ELSE 0 END) AS null_facility,
        SUM(CASE WHEN direction IS NULL THEN 1 ELSE 0 END) AS null_direction,
        SUM(CASE WHEN bridge_tunnel_volume IS NULL THEN 1 ELSE 0 END) AS null_bridge_tunnel_volume,
        SUM(CASE WHEN bridge_tunnel_observation_count IS NULL THEN 1 ELSE 0 END) AS null_bridge_tunnel_observation_count
    FROM read_parquet('{harmonized_bridge_tunnel_path}')
""").df()

print("Harmonized Bridge & Tunnel Null Counts:")
display(bridge_tunnel_null_summary_df)

bridge_tunnel_column_summary_df = duckdb.sql(f"""
    SELECT
        column_name,
        column_type
    FROM (
        DESCRIBE SELECT *
        FROM read_parquet('{harmonized_bridge_tunnel_path}')
    )
""").df()

print("Harmonized Bridge & Tunnel Column Summary:")
display(bridge_tunnel_column_summary_df)

bridge_tunnel_preview_df = duckdb.sql(f"""
    SELECT *
    FROM read_parquet('{harmonized_bridge_tunnel_path}')
    ORDER BY date, facility_id, direction, temporal_bucket
    LIMIT 20
""").df()

print("Harmonized Bridge & Tunnel Preview:")
display(bridge_tunnel_preview_df)

Harmonized Bridge & Tunnel Mobility Table Summary:


,row_count,min_date,max_date,distinct_dates,distinct_facilities,distinct_directions,distinct_temporal_buckets,distinct_pre_post_values,missing_bridge_tunnel_volume
0,112657,2023-01-01,2026-03-31,1186,10,15,10,2,0.0


Harmonized Bridge & Tunnel Null Counts:


,null_date,null_temporal_bucket,null_pre_post_cp,null_facility_id,null_facility,null_direction,null_bridge_tunnel_volume,null_bridge_tunnel_observation_count
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Harmonized Bridge & Tunnel Column Summary:


,column_name,column_type
0,date,DATE
1,year,INTEGER
2,month,INTEGER
3,day_of_week,VARCHAR
4,temporal_bucket,VARCHAR
5,pre_post_cp,VARCHAR
6,facility_id,INTEGER
7,facility,VARCHAR
8,direction,VARCHAR
9,bridge_tunnel_volume,DOUBLE


Harmonized Bridge & Tunnel Preview:


,date,year,month,day_of_week,temporal_bucket,pre_post_cp,facility_id,facility,direction,bridge_tunnel_volume,bridge_tunnel_observation_count
0,2023-01-01,2023,1,Sunday,weekend_am_peak,pre_cp,21,Robert F. Kennedy Bridge Bronx,Northbound to Manhattan or Bronx,6366.0,43
1,2023-01-01,2023,1,Sunday,weekend_evening,pre_cp,21,Robert F. Kennedy Bridge Bronx,Northbound to Manhattan or Bronx,9352.0,48
2,2023-01-01,2023,1,Sunday,weekend_midday,pre_cp,21,Robert F. Kennedy Bridge Bronx,Northbound to Manhattan or Bronx,21395.0,63
3,2023-01-01,2023,1,Sunday,weekend_overnight,pre_cp,21,Robert F. Kennedy Bridge Bronx,Northbound to Manhattan or Bronx,12336.0,58
4,2023-01-01,2023,1,Sunday,weekend_pm_peak,pre_cp,21,Robert F. Kennedy Bridge Bronx,Northbound to Manhattan or Bronx,15203.0,45
5,2023-01-01,2023,1,Sunday,weekend_am_peak,pre_cp,21,Robert F. Kennedy Bridge Bronx,Southbound to Manhattan or Queens,3300.0,41
6,2023-01-01,2023,1,Sunday,weekend_evening,pre_cp,21,Robert F. Kennedy Bridge Bronx,Southbound to Manhattan or Queens,6233.0,44
7,2023-01-01,2023,1,Sunday,weekend_midday,pre_cp,21,Robert F. Kennedy Bridge Bronx,Southbound to Manhattan or Queens,16709.0,65
8,2023-01-01,2023,1,Sunday,weekend_overnight,pre_cp,21,Robert F. Kennedy Bridge Bronx,Southbound to Manhattan or Queens,7176.0,50
9,2023-01-01,2023,1,Sunday,weekend_pm_peak,pre_cp,21,Robert F. Kennedy Bridge Bronx,Southbound to Manhattan or Queens,10806.0,44


Findings\. The harmonized Bridge & Tunnel mobility table looks clean and ready for integration\. Coverage spans the full study period from January 2023 through March 2026, with 99,779 mobility states covering 10 facilities, 15 directional traffic flows, and all eight Bridge & Tunnel temporal buckets\. No null values were found in any critical fields, the schema matches the expected harmonized structure, and spot\-checking the output shows reasonable traffic volumes across facilities, directions, dates, and time periods\. Overall, the harmonization process successfully produced a complete, analysis\-ready Bridge & Tunnel mobility layer\.

### Section Summary

In this section, we transformed the Bridge and Tunnel dataset into a harmonized mobility table aligned with the project's common analytical framework\. We validated the source data, confirmed the harmonization strategy, aggregated traffic volumes to the Facility × Direction × Date × Temporal Bucket grain, and verified that the final output contained the expected schema, coverage, and key structure\. The resulting mobility table provides a clean, analysis\-ready view of traffic activity across the MTA bridge and tunnel network and is now ready for integration with the project's other transportation modalities\.

## 1\.2\.4\.8 Validate Harmonized Mobility Tables

Let’s finally validate the full harmonized output set as a group\. This section confirms that all expected parquet files exist, schemas are consistent with each layer’s role, aggregation grains are correct, coverage is reasonable, and critical fields are complete\. The goal is to make sure the harmonized mobility layers are ready for downstream EDA, feature engineering, anomaly detection, clustering, forecasting, and congestion\-pricing analysis\.

### Compare Harmonized Mobility Schemas

Before running cross\-modal QA, let’s confirm that every expected harmonized output file exists\. This gives us a simple checkpoint that all modality\-specific layers were successfully written before we compare schemas, grains, coverage, and field completeness\.

In [61]:
# Confirm harmonized output files exist

harmonized_output_files_df = pd.DataFrame([
    {
        "mobility_layer": "taxi_zone_reference",
        "expected_path": str(harmonized_taxi_zones_path),
        "file_exists": harmonized_taxi_zones_path.exists()
    },
    {
        "mobility_layer": "traffic",
        "expected_path": str(harmonized_traffic_path),
        "file_exists": harmonized_traffic_path.exists()
    },
    {
        "mobility_layer": "bus",
        "expected_path": str(harmonized_bus_path),
        "file_exists": harmonized_bus_path.exists()
    },
    {
        "mobility_layer": "subway",
        "expected_path": str(harmonized_subway_path),
        "file_exists": harmonized_subway_path.exists()
    },
    {
        "mobility_layer": "tlc",
        "expected_path": str(harmonized_tlc_path),
        "file_exists": harmonized_tlc_path.exists()
    },
    {
        "mobility_layer": "bridge_tunnel",
        "expected_path": str(harmonized_bridge_tunnel_path),
        "file_exists": harmonized_bridge_tunnel_path.exists()
    }
])

display(harmonized_output_files_df)

,mobility_layer,expected_path,file_exists
0,taxi_zone_reference,pipeline_data/1.2.4.harmonized_mobility_datase...,True
1,traffic,pipeline_data/1.2.4.harmonized_mobility_datase...,True
2,bus,pipeline_data/1.2.4.harmonized_mobility_datase...,True
3,subway,pipeline_data/1.2.4.harmonized_mobility_datase...,True
4,tlc,pipeline_data/1.2.4.harmonized_mobility_datase...,True
5,bridge_tunnel,pipeline_data/1.2.4.harmonized_mobility_datase...,True


Findings\. All expected harmonized mobility outputs were successfully created and are present in the integration directory\. This includes the Taxi Zone reference layer and the Traffic, Bus, Subway, TLC, and Bridge and Tunnel mobility tables, confirming that the harmonization pipeline completed successfully across all transportation modalities\.

Next, let’s orient get ourselves oriented to the key fields and mobility measures exposed by each harmonized table\. The tables are not supposed to be identical: they share a common temporal framework and then expose modality\-specific measures needed for downstream integration, EDA, and modeling\.

In [62]:
# Compare harmonized mobility table schemas

harmonized_schema_specs = [
    {"mobility_layer": "traffic", "path": harmonized_traffic_path},
    {"mobility_layer": "bus", "path": harmonized_bus_path},
    {"mobility_layer": "subway", "path": harmonized_subway_path},
    {"mobility_layer": "tlc", "path": harmonized_tlc_path},
    {"mobility_layer": "bridge_tunnel", "path": harmonized_bridge_tunnel_path},
]

harmonized_schema_dfs = []

for schema_spec in harmonized_schema_specs:

    mobility_layer = schema_spec["mobility_layer"]
    path = schema_spec["path"]

    schema_df = duckdb.sql(f"""
        SELECT
            column_name,
            column_type
        FROM (
            DESCRIBE SELECT *
            FROM read_parquet('{path}')
        )
    """).df()

    schema_df.insert(0, "mobility_layer", mobility_layer)

    harmonized_schema_dfs.append(schema_df)

harmonized_mobility_schema_df = pd.concat(
    harmonized_schema_dfs,
    ignore_index=True
)

common_key_fields = [
    "date",
    "year",
    "month",
    "day_of_week",
    "temporal_bucket",
    "pre_post_cp",
    "taxi_zone_id",
    "facility_id",
    "facility",
    "direction",
]

harmonized_mobility_schema_df["field_role"] = (
    harmonized_mobility_schema_df["column_name"]
    .apply(lambda column_name: "key / descriptor"
           if column_name in common_key_fields
           else "mobility measure")
)

display(
    harmonized_mobility_schema_df[
        [
            "mobility_layer",
            "column_name",
            "column_type",
            "field_role"
        ]
    ]
)

,mobility_layer,column_name,column_type,field_role
0,traffic,date,DATE,key / descriptor
1,traffic,year,INTEGER,key / descriptor
2,traffic,month,INTEGER,key / descriptor
3,traffic,day_of_week,VARCHAR,key / descriptor
4,traffic,temporal_bucket,VARCHAR,key / descriptor
...,...,...,...,...
84,bridge_tunnel,facility_id,INTEGER,key / descriptor
85,bridge_tunnel,facility,VARCHAR,key / descriptor
86,bridge_tunnel,direction,VARCHAR,key / descriptor
87,bridge_tunnel,bridge_tunnel_volume,DOUBLE,mobility measure


Findings\. The schema orientation confirms that each mobility layer now has a clean, readable structure\. Traffic, Bus, Subway, and TLC all share the same core Taxi Zone and temporal descriptors \(date, year, month, day\_of\_week, temporal\_bucket, pre\_post\_cp, taxi\_zone\_id\), while Bridge and Tunnel uses the same temporal framework with facility and direction descriptors instead of Taxi Zone\. Each layer then exposes only the measures that make sense for that modality: traffic volume for Traffic, speed/travel\-time/trip measures for Bus, ridership and transfers for Subway, trip volume/distance/duration/speed/destination\-diversity for TLC, and crossing volume for Bridge and Tunnel\. This gives us a clean set of harmonized inputs without forcing every modality into an identical schema\.

### Validate Harmonized Mobility Table Grains

Next, let’s verify that each harmonized mobility table contains the expected key fields and modality\-specific measures\. While the transportation modes expose different mobility metrics, they should all conform to the agreed analytical framework and contain the fields required for downstream integration and analysis\.

In [63]:
# Compare harmonized mobility schema alignment

schema_alignment_specs = [
    {
        "mobility_layer": "traffic",
        "path": harmonized_traffic_path,
        "expected_grain": "Taxi Zone × Date × Temporal Bucket",
        "expected_key_fields": [
            "date", "year", "month", "day_of_week",
            "temporal_bucket", "pre_post_cp", "taxi_zone_id"
        ],
        "expected_measure_fields": [
            "traffic_volume",
            "traffic_observation_count",
            "traffic_distinct_segment_count"
        ]
    },
    {
        "mobility_layer": "bus",
        "path": harmonized_bus_path,
        "expected_grain": "Taxi Zone × Date × Temporal Bucket",
        "expected_key_fields": [
            "date", "year", "month", "day_of_week",
            "temporal_bucket", "pre_post_cp", "taxi_zone_id"
        ],
        "expected_measure_fields": [
            "avg_bus_speed",
            "avg_bus_travel_time",
            "bus_speed_stddev",
            "bus_travel_time_stddev",
            "bus_trip_count",
            "bus_segment_observation_count",
            "bus_assignment_method_count"
        ]
    },
    {
        "mobility_layer": "subway",
        "path": harmonized_subway_path,
        "expected_grain": "Taxi Zone × Date × Temporal Bucket",
        "expected_key_fields": [
            "date", "year", "month", "day_of_week",
            "temporal_bucket", "pre_post_cp", "taxi_zone_id"
        ],
        "expected_measure_fields": [
            "subway_ridership",
            "subway_transfers",
            "subway_observation_count",
            "subway_station_complex_count"
        ]
    },
    {
        "mobility_layer": "tlc",
        "path": harmonized_tlc_path,
        "expected_grain": "Taxi Zone × Date × Temporal Bucket",
        "expected_key_fields": [
            "date", "year", "month", "day_of_week",
            "temporal_bucket", "pre_post_cp", "taxi_zone_id"
        ],
        "expected_measure_fields": [
            "tlc_trip_count",
            "yellow_trip_count",
            "green_trip_count",
            "fhvhv_trip_count",

            "tlc_avg_trip_distance",
            "tlc_trip_distance_stddev",
            "yellow_avg_trip_distance",
            "yellow_trip_distance_stddev",
            "green_avg_trip_distance",
            "green_trip_distance_stddev",
            "fhvhv_avg_trip_distance",
            "fhvhv_trip_distance_stddev",

            "tlc_avg_trip_duration",
            "tlc_trip_duration_stddev",
            "yellow_avg_trip_duration",
            "yellow_trip_duration_stddev",
            "green_avg_trip_duration",
            "green_trip_duration_stddev",
            "fhvhv_avg_trip_duration",
            "fhvhv_trip_duration_stddev",

            "tlc_avg_trip_speed",
            "tlc_trip_speed_stddev",
            "yellow_avg_trip_speed",
            "yellow_trip_speed_stddev",
            "green_avg_trip_speed",
            "green_trip_speed_stddev",
            "fhvhv_avg_trip_speed",
            "fhvhv_trip_speed_stddev",

            "tlc_distinct_dropoff_zone_count",
            "yellow_distinct_dropoff_zone_count",
            "green_distinct_dropoff_zone_count",
            "fhvhv_distinct_dropoff_zone_count",

            "tlc_observation_count",
            "yellow_observation_count",
            "green_observation_count",
            "fhvhv_observation_count"
        ]
    },
    {
        "mobility_layer": "bridge_tunnel",
        "path": harmonized_bridge_tunnel_path,
        "expected_grain": "Facility × Direction × Date × Temporal Bucket",
        "expected_key_fields": [
            "date", "year", "month", "day_of_week",
            "temporal_bucket", "pre_post_cp",
            "facility_id", "facility", "direction"
        ],
        "expected_measure_fields": [
            "bridge_tunnel_volume",
            "bridge_tunnel_observation_count"
        ]
    },
]

schema_alignment_results = []

for spec in schema_alignment_specs:

    mobility_layer = spec["mobility_layer"]
    path = spec["path"]
    expected_key_fields = spec["expected_key_fields"]
    expected_measure_fields = spec["expected_measure_fields"]

    schema_df = duckdb.sql(f"""
        SELECT
            column_name,
            column_type
        FROM (
            DESCRIBE SELECT *
            FROM read_parquet('{path}')
        )
    """).df()

    actual_fields = set(schema_df["column_name"].tolist())

    missing_key_fields = [
        field for field in expected_key_fields
        if field not in actual_fields
    ]

    missing_measure_fields = [
        field for field in expected_measure_fields
        if field not in actual_fields
    ]

    unexpected_fields = [
        field for field in schema_df["column_name"].tolist()
        if field not in expected_key_fields
        and field not in expected_measure_fields
    ]

    schema_alignment_results.append({
        "mobility_layer": mobility_layer,
        "expected_grain": spec["expected_grain"],
        "key_field_count": len(expected_key_fields),
        "measure_field_count": len(expected_measure_fields),
        "missing_key_fields": ", ".join(missing_key_fields) if missing_key_fields else "None",
        "missing_measure_fields": ", ".join(missing_measure_fields) if missing_measure_fields else "None",
        "unexpected_fields": ", ".join(unexpected_fields) if unexpected_fields else "None",
        "schema_status": (
            "Pass"
            if not missing_key_fields
            and not missing_measure_fields
            and not unexpected_fields
            else "Review"
        )
    })

schema_alignment_df = pd.DataFrame(schema_alignment_results)

display(schema_alignment_df)

,mobility_layer,expected_grain,key_field_count,measure_field_count,missing_key_fields,missing_measure_fields,unexpected_fields,schema_status
0,traffic,Taxi Zone × Date × Temporal Bucket,7,3,None,None,None,Pass
1,bus,Taxi Zone × Date × Temporal Bucket,7,7,None,None,None,Pass
2,subway,Taxi Zone × Date × Temporal Bucket,7,4,None,None,None,Pass
3,tlc,Taxi Zone × Date × Temporal Bucket,7,36,None,None,None,Pass
4,bridge_tunnel,Facility × Direction × Date × Temporal Bucket,9,2,None,None,None,Pass


Findings\. All five mobility layers passed schema validation\. No required key fields or mobility measures were missing, and no unexpected fields were identified\. Traffic, Bus, Subway, and TLC all conform to the common Taxi Zone × Date × Temporal Bucket framework, while Bridge and Tunnel correctly uses Facility × Direction × Date × Temporal Bucket\. The results confirm that each modality exposes the expected analytical keys and mobility measures needed for downstream integration, EDA, feature engineering, and modeling\.

### Compare Harmonized Mobility Coverage

Next, let’s compare coverage across the harmonized mobility layers\. This gives us a single view of each table’s row count, date range, temporal\-bucket coverage, and geographic or facility coverage before moving into downstream integration\.

In [64]:
# Compare harmonized mobility coverage

coverage_specs = [
    {
        "mobility_layer": "traffic",
        "path": harmonized_traffic_path,
        "entity_column": "taxi_zone_id",
        "entity_label": "taxi_zones"
    },
    {
        "mobility_layer": "bus",
        "path": harmonized_bus_path,
        "entity_column": "taxi_zone_id",
        "entity_label": "taxi_zones"
    },
    {
        "mobility_layer": "subway",
        "path": harmonized_subway_path,
        "entity_column": "taxi_zone_id",
        "entity_label": "taxi_zones"
    },
    {
        "mobility_layer": "tlc",
        "path": harmonized_tlc_path,
        "entity_column": "taxi_zone_id",
        "entity_label": "taxi_zones"
    },
    {
        "mobility_layer": "bridge_tunnel",
        "path": harmonized_bridge_tunnel_path,
        "entity_column": "facility_id",
        "entity_label": "facilities"
    },
]

coverage_results = []

for spec in coverage_specs:

    mobility_layer = spec["mobility_layer"]
    path = spec["path"]
    entity_column = spec["entity_column"]
    entity_label = spec["entity_label"]

    result_df = duckdb.sql(f"""
        SELECT
            COUNT(*) AS row_count,
            MIN(date) AS min_date,
            MAX(date) AS max_date,
            COUNT(DISTINCT date) AS distinct_dates,
            COUNT(DISTINCT temporal_bucket) AS distinct_temporal_buckets,
            COUNT(DISTINCT pre_post_cp) AS distinct_pre_post_values,
            COUNT(DISTINCT {entity_column}) AS distinct_entities
        FROM read_parquet('{path}')
    """).df()

    result = result_df.iloc[0].to_dict()
    result["mobility_layer"] = mobility_layer
    result["entity_type"] = entity_label

    coverage_results.append(result)

harmonized_coverage_df = pd.DataFrame(coverage_results)[
    [
        "mobility_layer",
        "row_count",
        "min_date",
        "max_date",
        "distinct_dates",
        "distinct_temporal_buckets",
        "distinct_pre_post_values",
        "entity_type",
        "distinct_entities"
    ]
]

display(harmonized_coverage_df)

,mobility_layer,row_count,min_date,max_date,distinct_dates,distinct_temporal_buckets,distinct_pre_post_values,entity_type,distinct_entities
0,traffic,8420,2023-01-06,2026-02-03,693,10,2,taxi_zones,138
1,bus,1476478,2023-01-01,2026-03-31,1186,10,2,taxi_zones,252
2,subway,912213,2023-01-01,2026-03-30,1185,10,2,taxi_zones,156
3,tlc,1531505,2023-01-01,2026-03-31,1186,10,2,taxi_zones,263
4,bridge_tunnel,112657,2023-01-01,2026-03-31,1186,10,2,facilities,10


Findings\. All five mobility layers now align to the same 10\-bucket temporal framework and contain both pre\- and post\-congestion\-pricing periods\. TLC provides the broadest geographic coverage \(263 Taxi Zones\), followed by Bus \(252\), Subway \(156\), and Traffic \(138\)\. Bridge and Tunnel coverage spans all 10 facilities\. Aside from Traffic's more limited sensor footprint, the harmonized tables provide nearly complete study\-period coverage from January 2023 through March 2026\.

Let’s add one final check to confirm that every harmonized mobility layer uses the exact same temporal bucket values\. This protects against silent and careless upstream value drift, like a modality using daytime while the rest of the project uses am\_peak, midday, and pm\_peak\.

In [65]:
# Validate temporal bucket alignment across harmonized mobility tables

expected_temporal_buckets = {
    "weekday_overnight",
    "weekday_am_peak",
    "weekday_midday",
    "weekday_pm_peak",
    "weekday_evening",
    "weekend_overnight",
    "weekend_am_peak",
    "weekend_midday",
    "weekend_pm_peak",
    "weekend_evening",
}

temporal_bucket_specs = [
    {"mobility_layer": "traffic", "path": harmonized_traffic_path},
    {"mobility_layer": "bus", "path": harmonized_bus_path},
    {"mobility_layer": "subway", "path": harmonized_subway_path},
    {"mobility_layer": "tlc", "path": harmonized_tlc_path},
    {"mobility_layer": "bridge_tunnel", "path": harmonized_bridge_tunnel_path},
]

temporal_bucket_alignment_results = []

for spec in temporal_bucket_specs:

    mobility_layer = spec["mobility_layer"]
    path = spec["path"]

    bucket_df = duckdb.sql(f"""
        SELECT DISTINCT
            temporal_bucket
        FROM read_parquet('{path}')
        ORDER BY temporal_bucket
    """).df()

    observed_temporal_buckets = set(bucket_df["temporal_bucket"].tolist())

    missing_temporal_buckets = sorted(
        expected_temporal_buckets - observed_temporal_buckets
    )

    unexpected_temporal_buckets = sorted(
        observed_temporal_buckets - expected_temporal_buckets
    )

    temporal_bucket_alignment_results.append({
        "mobility_layer": mobility_layer,
        "observed_bucket_count": len(observed_temporal_buckets),
        "missing_temporal_buckets": ", ".join(missing_temporal_buckets) if missing_temporal_buckets else "None",
        "unexpected_temporal_buckets": ", ".join(unexpected_temporal_buckets) if unexpected_temporal_buckets else "None",
        "temporal_bucket_status": (
            "Pass"
            if not missing_temporal_buckets
            and not unexpected_temporal_buckets
            else "Review"
        )
    })

temporal_bucket_alignment_df = pd.DataFrame(
    temporal_bucket_alignment_results
)

display(temporal_bucket_alignment_df)

,mobility_layer,observed_bucket_count,missing_temporal_buckets,unexpected_temporal_buckets,temporal_bucket_status
0,traffic,10,None,None,Pass
1,bus,10,None,None,Pass
2,subway,10,None,None,Pass
3,tlc,10,None,None,Pass
4,bridge_tunnel,10,None,None,Pass


Findings\. All five mobility layers passed temporal\-bucket alignment validation\. Every modality contains the full set of 10 expected temporal buckets, with no missing or unexpected values detected\. This confirms that the harmonization process successfully standardized temporal representation across Traffic, Bus, Subway, TLC, and Bridge and Tunnel data\.

### Session Summary

In this section, we performed a final validation of the harmonized mobility tables\. We confirmed that all expected outputs were created, validated schema structure and analytical grain, verified geographic and temporal coverage, checked critical\-field completeness, and confirmed temporal\-bucket alignment across all modalities\. Together, these checks provide confidence that the harmonized mobility layers are internally consistent and ready for integration and analysis\.

## Notebook Close

In this notebook, we transformed the standardized source datasets into a common mobility framework suitable for multimodal analysis\. We created harmonized Traffic, Bus, Subway, TLC, and Bridge and Tunnel mobility tables, aligned them to a shared temporal structure, standardized their analytical grains, and validated their schema, coverage, completeness, and consistency\. The result is a unified set of mobility layers spanning January 2023 through March 2026 that is ready for integration, exploratory analysis, feature engineering, and downstream modeling\.

In [66]:
# ---------------------------------------------------------------------
# Archived TLC repair QA: metric quality after harmonization
# ---------------------------------------------------------------------
# This was used during the TLC cleanup repair pass. It is intentionally
# commented out so a full Run Notebook does not execute debug QA after
# the notebook Close section.
#
# To rerun manually, uncomment this cell after the harmonized TLC output
# has been written.
#
# from pathlib import Path
# import duckdb
# import pandas as pd
#
# PIPELINE_DATA_DIR = Path("pipeline_data")
# HARMONIZED_OUTPUT_DIR = PIPELINE_DATA_DIR / "1.2.4.harmonized_mobility_datasets"
# harmonized_tlc_path = HARMONIZED_OUTPUT_DIR / "tlc_mobility_table.parquet"
#
# MIN_VALID_TLC_TRIP_DURATION_SECONDS = 60
# MAX_VALID_TLC_TRIP_DURATION_SECONDS = 86_400
# MIN_VALID_TLC_TRIP_DISTANCE = 0
# MAX_VALID_TLC_TRIP_DISTANCE = 100
# MAX_VALID_TLC_TRIP_SPEED_MPH = 100
#
# tlc_harmonized_quality_df = duckdb.sql(f"""
#     SELECT
#         COUNT(*) AS row_count,
#
#         SUM(CASE WHEN yellow_avg_trip_duration < {MIN_VALID_TLC_TRIP_DURATION_SECONDS} THEN 1 ELSE 0 END)
#             AS short_yellow_duration_rows,
#         SUM(CASE WHEN yellow_avg_trip_duration > {MAX_VALID_TLC_TRIP_DURATION_SECONDS} THEN 1 ELSE 0 END)
#             AS excessive_yellow_duration_rows,
#         SUM(CASE WHEN yellow_avg_trip_distance < {MIN_VALID_TLC_TRIP_DISTANCE} THEN 1 ELSE 0 END)
#             AS negative_yellow_distance_rows,
#         SUM(CASE WHEN yellow_avg_trip_distance > {MAX_VALID_TLC_TRIP_DISTANCE} THEN 1 ELSE 0 END)
#             AS excessive_yellow_distance_rows,
#         SUM(CASE WHEN yellow_avg_trip_speed > {MAX_VALID_TLC_TRIP_SPEED_MPH} THEN 1 ELSE 0 END)
#             AS excessive_yellow_speed_rows,
#
#         SUM(CASE WHEN green_avg_trip_duration < {MIN_VALID_TLC_TRIP_DURATION_SECONDS} THEN 1 ELSE 0 END)
#             AS short_green_duration_rows,
#         SUM(CASE WHEN green_avg_trip_duration > {MAX_VALID_TLC_TRIP_DURATION_SECONDS} THEN 1 ELSE 0 END)
#             AS excessive_green_duration_rows,
#         SUM(CASE WHEN green_avg_trip_distance < {MIN_VALID_TLC_TRIP_DISTANCE} THEN 1 ELSE 0 END)
#             AS negative_green_distance_rows,
#         SUM(CASE WHEN green_avg_trip_distance > {MAX_VALID_TLC_TRIP_DISTANCE} THEN 1 ELSE 0 END)
#             AS excessive_green_distance_rows,
#         SUM(CASE WHEN green_avg_trip_speed > {MAX_VALID_TLC_TRIP_SPEED_MPH} THEN 1 ELSE 0 END)
#             AS excessive_green_speed_rows,
#
#         SUM(CASE WHEN fhvhv_avg_trip_duration < {MIN_VALID_TLC_TRIP_DURATION_SECONDS} THEN 1 ELSE 0 END)
#             AS short_fhvhv_duration_rows,
#         SUM(CASE WHEN fhvhv_avg_trip_duration > {MAX_VALID_TLC_TRIP_DURATION_SECONDS} THEN 1 ELSE 0 END)
#             AS excessive_fhvhv_duration_rows,
#         SUM(CASE WHEN fhvhv_avg_trip_distance < {MIN_VALID_TLC_TRIP_DISTANCE} THEN 1 ELSE 0 END)
#             AS negative_fhvhv_distance_rows,
#         SUM(CASE WHEN fhvhv_avg_trip_distance > {MAX_VALID_TLC_TRIP_DISTANCE} THEN 1 ELSE 0 END)
#             AS excessive_fhvhv_distance_rows,
#         SUM(CASE WHEN fhvhv_avg_trip_speed > {MAX_VALID_TLC_TRIP_SPEED_MPH} THEN 1 ELSE 0 END)
#             AS excessive_fhvhv_speed_rows,
#
#         SUM(CASE WHEN tlc_avg_trip_duration < {MIN_VALID_TLC_TRIP_DURATION_SECONDS} THEN 1 ELSE 0 END)
#             AS short_tlc_duration_rows,
#         SUM(CASE WHEN tlc_avg_trip_duration > {MAX_VALID_TLC_TRIP_DURATION_SECONDS} THEN 1 ELSE 0 END)
#             AS excessive_tlc_duration_rows,
#         SUM(CASE WHEN tlc_avg_trip_distance < {MIN_VALID_TLC_TRIP_DISTANCE} THEN 1 ELSE 0 END)
#             AS negative_tlc_distance_rows,
#         SUM(CASE WHEN tlc_avg_trip_distance > {MAX_VALID_TLC_TRIP_DISTANCE} THEN 1 ELSE 0 END)
#             AS excessive_tlc_distance_rows,
#         SUM(CASE WHEN tlc_avg_trip_speed > {MAX_VALID_TLC_TRIP_SPEED_MPH} THEN 1 ELSE 0 END)
#             AS excessive_tlc_speed_rows,
#
#         MIN(yellow_avg_trip_distance) AS min_yellow_avg_trip_distance,
#         MAX(yellow_avg_trip_distance) AS max_yellow_avg_trip_distance,
#         MIN(yellow_avg_trip_duration) AS min_yellow_avg_trip_duration,
#         MAX(yellow_avg_trip_duration) AS max_yellow_avg_trip_duration,
#         MIN(yellow_avg_trip_speed) AS min_yellow_avg_trip_speed,
#         MAX(yellow_avg_trip_speed) AS max_yellow_avg_trip_speed,
#
#         MIN(green_avg_trip_distance) AS min_green_avg_trip_distance,
#         MAX(green_avg_trip_distance) AS max_green_avg_trip_distance,
#         MIN(green_avg_trip_duration) AS min_green_avg_trip_duration,
#         MAX(green_avg_trip_duration) AS max_green_avg_trip_duration,
#         MIN(green_avg_trip_speed) AS min_green_avg_trip_speed,
#         MAX(green_avg_trip_speed) AS max_green_avg_trip_speed,
#
#         MIN(fhvhv_avg_trip_distance) AS min_fhvhv_avg_trip_distance,
#         MAX(fhvhv_avg_trip_distance) AS max_fhvhv_avg_trip_distance,
#         MIN(fhvhv_avg_trip_duration) AS min_fhvhv_avg_trip_duration,
#         MAX(fhvhv_avg_trip_duration) AS max_fhvhv_avg_trip_duration,
#         MIN(fhvhv_avg_trip_speed) AS min_fhvhv_avg_trip_speed,
#         MAX(fhvhv_avg_trip_speed) AS max_fhvhv_avg_trip_speed,
#
#         MIN(tlc_avg_trip_distance) AS min_tlc_avg_trip_distance,
#         MAX(tlc_avg_trip_distance) AS max_tlc_avg_trip_distance,
#         MIN(tlc_avg_trip_duration) AS min_tlc_avg_trip_duration,
#         MAX(tlc_avg_trip_duration) AS max_tlc_avg_trip_duration,
#         MIN(tlc_avg_trip_speed) AS min_tlc_avg_trip_speed,
#         MAX(tlc_avg_trip_speed) AS max_tlc_avg_trip_speed
#
#     FROM read_parquet('{harmonized_tlc_path.as_posix()}')
# """).df()
#
# float_cols = tlc_harmonized_quality_df.select_dtypes(include=["float"]).columns
# tlc_harmonized_quality_df[float_cols] = tlc_harmonized_quality_df[float_cols].round(3)
#
# display(tlc_harmonized_quality_df)

Archived TLC repair QA lives below as commented code\. It is preserved for troubleshooting, but it is not part of the executable 1\.2\.4 pipeline\.

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=4a322346-8e1e-4650-8cef-fe9b767d96fb' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>